# New York Quarterly Emergency Department Demand Forecasting
**DATA 975 Capstone — Thi Thanh Hien Bui (Emi)**

This notebook forecasts **total emergency department encounters by facility county and quarter** for staffing and capacity-planning analysis.

The notebook follows the same **Phase → Task → weekly deliverable** structure used in the annual project **through July 28**, while retaining the quarterly data logic:

- `target_lag1` is the previous quarter;
- `target_lag4` is the same quarter in the previous year;
- the change target is current encounters minus `target_lag4`;
- ACS predictors use a two-calendar-year lag so the public 5-year estimates would be available before the target year;
- weather predictors come from the same quarter in the previous year;
- rolling validation holds out one complete calendar quarter across all eligible facility counties at a time;
- the final holdout is the latest sufficiently complete calendar year.

The forecasting target is raw encounter volume. A population-standardized proxy is used only for descriptive demographic analysis because encounters are assigned to **facility county**, not necessarily the patient's county of residence.

This is a retrospective forecasting prototype based on public data. A live operational deployment would require a current internal encounter feed.

### Weekly structure

- **Through July 7:** Phases 1–5, Tasks 1–4 — configuration, acquisition, cleaning, EDA, and leakage-safe feature engineering.
- **Week of July 14:** Task 5 — persistence benchmarks and compact candidate-model development.
- **Week of July 21:** Task 6 — complete-quarter rolling-origin validation and stability analysis.
- **Week of July 28:** Tasks 7–10 — feature pruning and tuning, locked holdout evaluation, interpretation, artifact creation, inference validation, and final reporting.


## 0 · Setup and imports

Load the scientific-Python stack, create portable project folders, and define the processed-data, report, image, and model-artifact paths used throughout the notebook.


In [ ]:
from pathlib import Path
import io
import math
import os
import re
import sys
import time
import warnings
import zipfile
import subprocess
import sklearn

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    import xgboost as xgb
    from xgboost import XGBRegressor

try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])
    import shap

from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/MS DATA SCIENCE/Capstone975")
except ImportError:
    PROJECT_DIR = Path.cwd() / "Capstone975"

RAW_DIR = PROJECT_DIR / "data/raw"
PROCESSED_DIR = PROJECT_DIR / "data/processed"
IMAGE_DIR = PROJECT_DIR / "data/images"
MODEL_DIR = PROJECT_DIR / "model"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [RAW_DIR, PROCESSED_DIR, IMAGE_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ANALYSIS_TABLE_PATH = PROCESSED_DIR / "county_quarter_analysis.csv"
FACILITY_TABLE_PATH = PROCESSED_DIR / "facility_county_quarter.csv"
CENSUS_TABLE_PATH = PROCESSED_DIR / "census_county_year.csv"
WEATHER_TABLE_PATH = PROCESSED_DIR / "weather_county_quarter.csv"

print("Project folder:", PROJECT_DIR)
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("SHAP:", shap.__version__)


## 1 · Phase 1 — Configuration (quarterly scoping decisions)

The quarterly project is defined by the following decisions:

- **Geography:** New York facility county;
- **Unit of analysis:** one facility county-quarter;
- **Target:** total emergency department encounters;
- **Temporal history:** previous quarter (`target_lag1`) and same quarter last year (`target_lag4`);
- **External predictors:** two-year-lag ACS and same-quarter prior-year weather;
- **Validation:** expanding-window rolling origin by complete calendar quarter;
- **Final holdout:** the latest year with four sufficiently complete quarters and aligned predictor coverage.

The final submission uses the saved local SPARCS snapshot so the results remain reproducible. If the local file does not exist, the notebook downloads it once. ACS and weather add only source periods that are not already present.


In [ ]:
STATE_USPS = "NY"
STATE_FIPS = "36"
COVID_YEARS = [2020, 2021]

REFRESH_FACILITY_SOURCE = False
FORCE_REBUILD_RAW_FILES = False
MIN_TRAIN_QUARTERS = 8

SOCRATA_DOMAIN = "health.data.ny.gov"
SPARCS_RESOURCE_ID = "5gzv-zv2z"
CENSUS_DATASET = "acs/acs5"
OPEN_METEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
GAZETTEER_URL = (
    "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/"
    "2023_Gazetteer/2023_Gaz_counties_national.zip"
)

TARGET = "total_ed_encounters"

CENSUS_VARIABLES = {
    "pop": "B01003_001E",
    "median_income": "B19013_001E",
    "pov_total": "B17001_001E",
    "pov_below": "B17001_002E",
    "age_total": "B01001_001E",
    "male_under5": "B01001_003E",
    "female_under5": "B01001_027E",
}
AGE_65PLUS_VARS = [
    "B01001_020E", "B01001_021E", "B01001_022E",
    "B01001_023E", "B01001_024E", "B01001_025E",
    "B01001_044E", "B01001_045E", "B01001_046E",
    "B01001_047E", "B01001_048E", "B01001_049E",
]

# Keep the API key outside the notebook before sharing.
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")

print("Forecasting target:", TARGET)


# Phase 2 — Data Collection & Acquisition

## Task 1 — Data Collection & Acquisition

The quarterly analysis combines four public sources at different grains. Their roles and alignment rules are documented before acquisition.

| Source | Raw grain | Role in the quarterly project |
|---|---|---|
| SPARCS ED Encounters by Facility | facility-quarter plus statewide summaries | forecasting target, demand history, statewide context |
| Census Gazetteer | county | county FIPS, county names, and centroids |
| ACS 5-Year | county-year | two-year-lag demographic predictors |
| Open-Meteo archive | county centroid-day | same-quarter previous-year weather predictors |

**Task 1 deliverable:** verify that each source loads, document its provenance and grain, and confirm that the temporal alignment supports a facility county-quarter model without using future information.


## 2 · Shared helper functions

These reusable functions standardize column names, download public data, calculate metrics and diagnostics, create regional labels, and support the later modeling workflow.


In [ ]:
def clean_column_names(df):
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.lower()
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return out


def to_num(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )


def name_key(value):
    text = str(value).lower().split(",")[0]
    text = text.replace("county", "").replace(".", "").replace("'", "")
    return re.sub(r"\s+", " ", text).strip()


def download_socrata(resource_id, limit=50000):
    url = f"https://{SOCRATA_DOMAIN}/resource/{resource_id}.json"
    frames = []
    offset = 0

    while True:
        response = requests.get(
            url,
            params={"$limit": limit, "$offset": offset},
            timeout=90,
        )
        response.raise_for_status()
        rows = response.json()

        if not rows:
            break

        frames.append(pd.DataFrame(rows))

        if len(rows) < limit:
            break

        offset += limit
        time.sleep(0.2)

    if not frames:
        raise RuntimeError("The Socrata request returned no rows.")

    return pd.concat(frames, ignore_index=True)

def format_count(value):
    """Format encounter counts and other whole-number chart labels."""
    if pd.isna(value):
        return ""
    return f"{float(value):,.0f}"


def format_decimal(value, digits=2):
    """Format continuous chart values without unnecessary scientific notation."""
    if pd.isna(value):
        return ""
    return f"{float(value):,.{digits}f}"


def add_line_value_labels(
    ax,
    x_values,
    y_values,
    formatter=format_count,
    fontsize=7,
    offset_points=7,
):
    """Place a readable value above every point in a line chart."""
    for x_value, y_value in zip(x_values, y_values):
        if pd.isna(y_value):
            continue
        ax.annotate(
            formatter(y_value),
            xy=(x_value, y_value),
            xytext=(0, offset_points),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=fontsize,
        )


def add_bar_value_labels(
    ax,
    bars,
    formatter=format_count,
    horizontal=False,
    fontsize=8,
):
    """Label every bar using its displayed value."""
    values = [
        bar.get_width() if horizontal else bar.get_height()
        for bar in bars
    ]
    labels = [formatter(value) for value in values]
    ax.bar_label(
        bars,
        labels=labels,
        padding=3,
        fontsize=fontsize,
        label_type="edge",
    )


def add_histogram_count_labels(ax, patches, fontsize=7):
    """Show the frequency above each nonempty histogram bin."""
    for patch in patches:
        count = patch.get_height()
        if count <= 0:
            continue
        ax.annotate(
            f"{int(round(count)):,}",
            xy=(
                patch.get_x() + patch.get_width() / 2,
                count,
            ),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=fontsize,
            rotation=90,
        )


def add_selected_scatter_labels(
    ax,
    frame,
    x_column,
    y_column,
    label_column=None,
    score_column=None,
    n_labels=10,
    x_formatter=lambda value: format_decimal(value, 1),
    y_formatter=lambda value: format_decimal(value, 1),
    note="Labels show the 10 most informative observations.",
):
    """
    Label a readable subset of a dense scatterplot.

    By default, the largest y-values are labeled. When score_column is given,
    observations with the largest absolute score are labeled instead.
    """
    required = [x_column, y_column]
    if label_column is not None:
        required.append(label_column)
    if score_column is not None and score_column not in required:
        required.append(score_column)

    valid = frame[required].dropna(subset=[x_column, y_column]).copy()
    if valid.empty:
        return

    if score_column is None:
        valid["_label_score"] = valid[y_column]
    else:
        valid["_label_score"] = valid[score_column].abs()

    selected = valid.nlargest(min(n_labels, len(valid)), "_label_score")

    offsets = [
        (5, 5),
        (5, -12),
        (-5, 5),
        (-5, -12),
    ]

    for position, (_, row) in enumerate(selected.iterrows()):
        prefix = ""
        if label_column is not None and pd.notna(row[label_column]):
            prefix = f"{row[label_column]}\n"

        label = (
            f"{prefix}"
            f"({x_formatter(row[x_column])}, "
            f"{y_formatter(row[y_column])})"
        )
        offset_x, offset_y = offsets[position % len(offsets)]
        ax.annotate(
            label,
            xy=(row[x_column], row[y_column]),
            xytext=(offset_x, offset_y),
            textcoords="offset points",
            ha="left" if offset_x >= 0 else "right",
            va="bottom" if offset_y >= 0 else "top",
            fontsize=7,
            alpha=0.85,
        )

    ax.text(
        0.01,
        0.99,
        note,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        alpha=0.7,
    )


## 3 · Phase 2 — Acquire NY ED data (SPARCS facility-quarter)

Download the current SPARCS ED Encounters by Facility table. Preserve both facility-quarter rows and official statewide summary rows. The facility-quarter rows later form the county-quarter forecasting target; statewide rows are retained only for descriptive context.


In [ ]:
FACILITY_RAW_PATH = RAW_DIR / "sparcs_ed_facility_quarter.csv"


def load_facility_source():
    """Load the saved SPARCS snapshot, downloading it only when needed."""
    if REFRESH_FACILITY_SOURCE or FORCE_REBUILD_RAW_FILES or not FACILITY_RAW_PATH.exists():
        try:
            raw = download_socrata(SPARCS_RESOURCE_ID)
            raw.to_csv(FACILITY_RAW_PATH, index=False)
            print("Downloaded the current SPARCS facility source.")
        except Exception as error:
            if not FACILITY_RAW_PATH.exists():
                raise
            print("SPARCS update check failed:", error)
            print("Using the local SPARCS raw file.")
            raw = pd.read_csv(FACILITY_RAW_PATH, dtype=str)
    else:
        raw = pd.read_csv(FACILITY_RAW_PATH, dtype=str)
        print("Loaded the local SPARCS raw file.")

    raw = clean_column_names(raw)

    required = {
        "year",
        "quarter",
        "facility_id",
        "facility_name",
        "total_ed_encounters",
        "facility_county_fips",
    }
    missing = required - set(raw.columns)
    if missing:
        raise KeyError(f"SPARCS is missing columns: {sorted(missing)}")

    actual_facility_rows = (
        raw["facility_county_fips"].notna()
        & raw["facility_id"].astype(str).ne("0")
    )
    if actual_facility_rows.sum() == 0:
        raise ValueError("No facility-county rows were found.")

    return raw


facility_raw = load_facility_source()

facility_quarter_rows = (
    facility_raw["facility_county_fips"].notna()
    & facility_raw["facility_id"].astype(str).ne("0")
    & facility_raw["quarter"].astype(str).str.upper().isin(["Q1", "Q2", "Q3", "Q4"])
)

facility_years = (
    to_num(facility_raw.loc[facility_quarter_rows, "year"])
    .dropna()
    .astype(int)
)
FACILITY_START_YEAR = int(facility_years.min())
FACILITY_END_YEAR = int(facility_years.max())

# ACS uses a conservative two-year lag; weather uses a one-year lag.
ACS_SOURCE_YEARS = list(range(FACILITY_START_YEAR - 2, FACILITY_END_YEAR - 1))
WEATHER_SOURCE_YEARS = list(range(FACILITY_START_YEAR - 1, FACILITY_END_YEAR))
DATA_AS_OF_DATE = pd.Timestamp.fromtimestamp(
    FACILITY_RAW_PATH.stat().st_mtime
).date().isoformat()

print("SPARCS rows:", len(facility_raw))
print("Facility target years:", FACILITY_START_YEAR, "to", FACILITY_END_YEAR)
print("ACS source years:", min(ACS_SOURCE_YEARS), "to", max(ACS_SOURCE_YEARS))
print("Weather source years:", min(WEATHER_SOURCE_YEARS), "to", max(WEATHER_SOURCE_YEARS))
print("SPARCS snapshot date:", DATA_AS_OF_DATE)

display(
    facility_raw.loc[
        facility_quarter_rows,
        [
            "year",
            "quarter",
            "facility_id",
            "facility_name",
            "facility_county_fips",
            "facility_county",
            "total_ed_encounters",
        ],
    ].head()
)


## 4 · Phase 2 — Acquire New York county geography

Load the county FIPS crosswalk and county centroids from the Census Gazetteer. FIPS is the stable join key used across SPARCS, ACS, weather, and geographic features.


In [ ]:
COUNTY_GEO_PATH = RAW_DIR / "ny_county_geo.csv"

if COUNTY_GEO_PATH.exists() and not FORCE_REBUILD_RAW_FILES:
    county_geo = pd.read_csv(COUNTY_GEO_PATH, dtype={"fips": str})
else:
    response = requests.get(GAZETTEER_URL, timeout=90)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        text_name = next(name for name in zf.namelist() if name.lower().endswith(".txt"))
        geo_raw = pd.read_csv(
            io.StringIO(zf.read(text_name).decode("latin-1")),
            sep="\t",
            dtype=str,
        )

    geo_raw.columns = [column.strip() for column in geo_raw.columns]
    geo_raw = geo_raw.loc[geo_raw["USPS"].eq(STATE_USPS)].copy()

    county_geo = pd.DataFrame({
        "fips": geo_raw["GEOID"].str.strip(),
        "county_name": geo_raw["NAME"].str.strip(),
        "lat": geo_raw["INTPTLAT"].str.strip().astype(float),
        "lon": geo_raw["INTPTLONG"].str.strip().astype(float),
    })
    county_geo["name_key"] = county_geo["county_name"].map(name_key)
    county_geo.to_csv(COUNTY_GEO_PATH, index=False)

county_geo["fips"] = county_geo["fips"].astype(str).str.zfill(5)

print("New York counties:", len(county_geo))
display(county_geo.head())


## 5 · Phase 2 — Acquire Census ACS demographics

Pull ACS 5-Year county estimates for New York. During panel construction, each ACS year is shifted forward by two target years. This simple conservative lag avoids using a public ACS vintage before it would normally be available.


In [ ]:
CENSUS_RAW_PATH = RAW_DIR / "census_acs_county.csv"


def download_acs_year(year):
    variables = list(CENSUS_VARIABLES.values()) + AGE_65PLUS_VARS
    url = f"https://api.census.gov/data/{year}/{CENSUS_DATASET}"
    params = {
        "get": "NAME," + ",".join(variables),
        "for": "county:*",
        "in": f"state:{STATE_FIPS}",
    }
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY

    response = requests.get(url, params=params, timeout=60)
    if response.status_code != 200:
        print(f"ACS {year} is not available.")
        return pd.DataFrame()

    rows = response.json()
    frame = pd.DataFrame(rows[1:], columns=rows[0])
    frame["year"] = year
    return frame


if CENSUS_RAW_PATH.exists() and not FORCE_REBUILD_RAW_FILES:
    census_raw = pd.read_csv(
        CENSUS_RAW_PATH,
        dtype={"state": str, "county": str},
    )
else:
    census_raw = pd.DataFrame()

saved_acs_years = set()
if not census_raw.empty and "year" in census_raw:
    saved_acs_years = set(to_num(census_raw["year"]).dropna().astype(int))

missing_acs_years = [year for year in ACS_SOURCE_YEARS if year not in saved_acs_years]
new_acs_frames = [download_acs_year(year) for year in missing_acs_years]
new_acs_frames = [frame for frame in new_acs_frames if not frame.empty]

if new_acs_frames:
    census_raw = pd.concat([census_raw] + new_acs_frames, ignore_index=True)

if census_raw.empty:
    raise RuntimeError("No ACS data are available.")

census_raw["state"] = census_raw["state"].astype(str).str.zfill(2)
census_raw["county"] = census_raw["county"].astype(str).str.zfill(3)
census_raw["year"] = to_num(census_raw["year"]).astype("Int64")
census_raw = (
    census_raw.drop_duplicates(["state", "county", "year"], keep="last")
    .sort_values(["year", "state", "county"])
    .reset_index(drop=True)
)
census_raw.to_csv(CENSUS_RAW_PATH, index=False)

print("ACS years:", int(census_raw["year"].min()), "to", int(census_raw["year"].max()))


## 6 · Phase 2 — Acquire daily county weather

Download daily historical weather at each county centroid. Daily observations are aggregated to facility county-quarter and shifted by four quarters so each target quarter uses weather from the same quarter of the previous year.


In [ ]:
WEATHER_RAW_PATH = RAW_DIR / "weather_daily_by_county.csv"


def download_weather(row, start_date, end_date):
    params = {
        "latitude": row["lat"],
        "longitude": row["lon"],
        "start_date": start_date,
        "end_date": end_date,
        "daily": (
            "temperature_2m_max,temperature_2m_min,"
            "precipitation_sum,snowfall_sum"
        ),
        "temperature_unit": "fahrenheit",
        "precipitation_unit": "inch",
        "timezone": "America/New_York",
    }

    response = requests.get(OPEN_METEO_ARCHIVE, params=params, timeout=60)
    response.raise_for_status()
    daily = response.json()["daily"]

    return pd.DataFrame({
        "fips": str(row["fips"]).zfill(5),
        "date": daily["time"],
        "tmax": daily["temperature_2m_max"],
        "tmin": daily["temperature_2m_min"],
        "precip": daily["precipitation_sum"],
        "snowfall": daily.get("snowfall_sum", [np.nan] * len(daily["time"])),
    })


if WEATHER_RAW_PATH.exists() and not FORCE_REBUILD_RAW_FILES:
    weather_daily = pd.read_csv(WEATHER_RAW_PATH, dtype={"fips": str})
    weather_daily["date"] = pd.to_datetime(weather_daily["date"], errors="coerce")
else:
    weather_daily = pd.DataFrame(
        columns=["fips", "date", "tmax", "tmin", "precip", "snowfall"]
    )

required_start = pd.Timestamp(f"{min(WEATHER_SOURCE_YEARS)}-01-01")
required_end = pd.Timestamp(f"{max(WEATHER_SOURCE_YEARS)}-12-31")
weather_additions = []

for row_number, row in county_geo.iterrows():
    fips = str(row["fips"]).zfill(5)
    county_saved = weather_daily.loc[weather_daily["fips"].astype(str).str.zfill(5).eq(fips)]

    if county_saved.empty:
        start_date = required_start
    else:
        start_date = max(required_start, county_saved["date"].max() + pd.Timedelta(days=1))

    if start_date > required_end:
        continue

    print(
        f"Weather {row_number + 1}/{len(county_geo)} | {fips} | "
        f"{start_date.date()} to {required_end.date()}"
    )
    weather_additions.append(
        download_weather(row, str(start_date.date()), str(required_end.date()))
    )
    time.sleep(1)

if weather_additions:
    weather_daily = pd.concat(
        [weather_daily, pd.concat(weather_additions, ignore_index=True)],
        ignore_index=True,
    )

if weather_daily.empty:
    raise RuntimeError("No weather data are available.")

weather_daily["fips"] = weather_daily["fips"].astype(str).str.zfill(5)
weather_daily["date"] = pd.to_datetime(weather_daily["date"], errors="coerce")
weather_daily = (
    weather_daily.dropna(subset=["fips", "date"])
    .drop_duplicates(["fips", "date"], keep="last")
    .sort_values(["fips", "date"])
    .reset_index(drop=True)
)
weather_daily.assign(date=weather_daily["date"].dt.strftime("%Y-%m-%d")).to_csv(
    WEATHER_RAW_PATH,
    index=False,
)

print("Weather dates:", weather_daily["date"].min().date(), "to", weather_daily["date"].max().date())


### Task 1 note

The quarterly target comes directly from SPARCS facility-quarter encounter counts. Census Gazetteer supplies the common county FIPS and centroids, ACS supplies two-year-lag demographic context, and Open-Meteo supplies same-quarter previous-year weather. The raw grains are preserved before aggregation so provenance and temporal alignment remain auditable.


# Phase 3 — Data Cleaning, Preprocessing, and Panel Construction

## Task 2 — Data Cleaning & Preprocessing

This phase converts the raw sources into one auditable row per facility county-quarter. It verifies data types and keys, handles unavailable or suppressed values explicitly, creates exact temporal lags, aligns lagged external predictors, and checks coverage before modeling.


## 7 · Clean and aggregate facility encounters by facility county-quarter

Statewide summary rows remain available in `facility_raw` for the project introduction. The modeling table uses only actual facility rows with a county FIPS and aggregates them to one row per facility county and quarter.

Suppressed or nonnumeric target cells cannot be included in the public county total. Their count is reported explicitly, and totals in affected facility county-quarters may therefore be understated by an unknown amount.


In [ ]:
ENCOUNTER_COLUMNS = [
    "total_ed_encounters",
    "treat_and_release_ed",
    "ed_encounters_requiring",
    "ed_encounters_admitted_as",
]


def sum_with_missing(series):
    return series.sum(min_count=1)


def prepare_facility_quarter(raw):
    df = clean_column_names(raw)
    df["year"] = to_num(df["year"]).astype("Int64")
    df["quarter"] = df["quarter"].astype(str).str.upper().str.strip()
    df["fips"] = (
        to_num(df["facility_county_fips"])
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.zfill(5)
    )

    df = df.loc[
        df["quarter"].isin(["Q1", "Q2", "Q3", "Q4"])
        & df["fips"].isin(set(county_geo["fips"]))
    ].copy()

    for column in ENCOUNTER_COLUMNS:
        if column not in df:
            df[column] = np.nan
        df[column] = to_num(df[column])

    suppressed_target_rows = int(df[TARGET].isna().sum())

    df = df.dropna(subset=["year"]).copy()
    df["year"] = df["year"].astype(int)
    df["quarter_num"] = df["quarter"].str[-1].astype(int)
    df["period_index"] = df["year"] * 4 + df["quarter_num"]

    group_keys = ["fips", "year", "quarter", "quarter_num", "period_index"]

    facility_counts = (
        df.groupby(group_keys, as_index=False)["facility_id"]
        .nunique()
        .rename(columns={"facility_id": "facility_count"})
    )

    numeric_rows = df.dropna(subset=[TARGET]).copy()
    county_quarter = (
        numeric_rows.groupby(group_keys, as_index=False)
        .agg({
            TARGET: sum_with_missing,
            "treat_and_release_ed": sum_with_missing,
            "ed_encounters_requiring": sum_with_missing,
            "ed_encounters_admitted_as": sum_with_missing,
        })
        .merge(facility_counts, on=group_keys, how="left", validate="one_to_one")
    )

    components = county_quarter[
        [
            "treat_and_release_ed",
            "ed_encounters_requiring",
            "ed_encounters_admitted_as",
        ]
    ].sum(axis=1, min_count=1)
    county_quarter["component_gap"] = county_quarter[TARGET] - components

    county_quarter = county_quarter.merge(
        county_geo[["fips", "county_name", "lat", "lon"]],
        on="fips",
        how="left",
        validate="many_to_one",
    )

    return (
        county_quarter.sort_values(["fips", "period_index"]).reset_index(drop=True),
        suppressed_target_rows,
    )


facility_panel, SUPPRESSED_TARGET_ROWS = prepare_facility_quarter(facility_raw)
facility_panel.to_csv(FACILITY_TABLE_PATH, index=False)

print("Facility county-quarter table:", facility_panel.shape)
print("Counties represented:", facility_panel["fips"].nunique())
print("Facility-quarter rows with suppressed or nonnumeric target:", SUPPRESSED_TARGET_ROWS)

display(
    facility_panel[
        ["county_name", "fips", "year", "quarter", TARGET, "facility_count"]
    ].head(10)
)


## 8 · Clean ACS and aggregate weather to county-quarter

Convert ACS API fields to numeric values, treat unavailable negative ACS sentinels as missing, derive demographic rates, and aggregate daily weather into quarterly temperature, precipitation, snowfall, and extreme-day summaries.


In [ ]:
def prepare_census(raw):
    df = raw.copy()
    df["fips"] = (
        df["state"].astype(str).str.zfill(2)
        + df["county"].astype(str).str.zfill(3)
    )
    df["year"] = to_num(df["year"]).astype("Int64")

    variables = list(CENSUS_VARIABLES.values()) + AGE_65PLUS_VARS
    for variable in variables:
        df[variable] = to_num(df[variable])
        df.loc[df[variable] < 0, variable] = np.nan

    age_total = df[CENSUS_VARIABLES["age_total"]]
    df["pop"] = df[CENSUS_VARIABLES["pop"]]
    df["median_income"] = df[CENSUS_VARIABLES["median_income"]]
    df["poverty_rate"] = (
        df[CENSUS_VARIABLES["pov_below"]]
        / df[CENSUS_VARIABLES["pov_total"]]
        * 100
    )
    df["pct_under5"] = (
        df[CENSUS_VARIABLES["male_under5"]]
        + df[CENSUS_VARIABLES["female_under5"]]
    ) / age_total * 100
    df["pct_65plus"] = df[AGE_65PLUS_VARS].sum(axis=1) / age_total * 100

    keep = [
        "fips",
        "year",
        "pop",
        "median_income",
        "poverty_rate",
        "pct_under5",
        "pct_65plus",
    ]
    return df[keep].dropna(subset=["fips", "year"]).reset_index(drop=True)


def aggregate_weather_quarter(daily):
    df = daily.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["fips", "date"]).copy()

    for column in ["tmax", "tmin", "precip", "snowfall"]:
        df[column] = to_num(df[column])

    df["year"] = df["date"].dt.year
    df["quarter_num"] = df["date"].dt.quarter
    df["quarter"] = "Q" + df["quarter_num"].astype(str)
    df["tavg"] = (df["tmax"] + df["tmin"]) / 2
    df["hot_day"] = (df["tmax"] >= 90).astype(int)
    df["freeze_day"] = (df["tmin"] <= 32).astype(int)
    df["wet_day"] = (df["precip"] >= 0.01).astype(int)

    return (
        df.groupby(["fips", "year", "quarter", "quarter_num"], as_index=False)
        .agg(
            weather_days_observed=("date", "nunique"),
            tavg_mean=("tavg", "mean"),
            precip_total=("precip", "sum"),
            snowfall_total=("snowfall", "sum"),
            hot_days=("hot_day", "sum"),
            freeze_days=("freeze_day", "sum"),
            wet_days=("wet_day", "sum"),
        )
    )


census_county_year = prepare_census(census_raw)
weather_county_quarter = aggregate_weather_quarter(weather_daily)

census_county_year.to_csv(CENSUS_TABLE_PATH, index=False)
weather_county_quarter.to_csv(WEATHER_TABLE_PATH, index=False)

print("ACS prepared table:", census_county_year.shape)
print("Weather prepared table:", weather_county_quarter.shape)


## 9 · Build the facility county-quarter analytical panel

The panel is built in three transparent stages. Each DataFrame name reflects its role and is used only once in the sequence.


### 9.1 · Engineer exact demand-history features

`target_lag1` is the previous quarter. `target_lag4` is the same quarter in the previous year. Exact lags are joined by facility-county FIPS and quarter index, so a missing quarter remains missing rather than shifting the sequence incorrectly.

`target_roll4` is retained for descriptive interpretation but is later excluded from the candidate feature set because it is an exact average of the four quarterly lag columns.


In [ ]:
def add_exact_lag(frame, source_column, lag_periods, output_column):
    lookup = frame[["fips", "period_index", source_column]].copy()
    lookup["period_index"] = lookup["period_index"] + lag_periods
    lookup = lookup.rename(columns={source_column: output_column})

    return frame.merge(
        lookup,
        on=["fips", "period_index"],
        how="left",
        validate="one_to_one",
    )


demand_history_panel = facility_panel.copy()

for lag in range(1, 5):
    demand_history_panel = add_exact_lag(
        demand_history_panel,
        source_column=TARGET,
        lag_periods=lag,
        output_column=f"target_lag{lag}",
    )

demand_history_panel["target_roll4"] = demand_history_panel[
    ["target_lag1", "target_lag2", "target_lag3", "target_lag4"]
].mean(axis=1, skipna=False)

demand_history_panel = add_exact_lag(
    demand_history_panel,
    source_column="facility_count",
    lag_periods=1,
    output_column="facility_count_lag1",
)

example_fips = demand_history_panel["fips"].dropna().iloc[0]
example_name = demand_history_panel.loc[
    demand_history_panel["fips"].eq(example_fips),
    "county_name",
].iloc[0]

print("Lag example for:", example_name)
display(
    demand_history_panel.loc[
        demand_history_panel["fips"].eq(example_fips),
        [
            "county_name",
            "year",
            "quarter",
            TARGET,
            "target_lag1",
            "target_lag4",
            "target_roll4",
        ],
    ].head(12)
)


### 9.2 · Align two-year-lag ACS and prior-year weather

The source year is preserved for auditing. ACS is shifted forward by two target years, while weather is shifted forward by one year. For example, 2022 ACS and 2023 Q2 weather support the 2024 Q2 target.


In [ ]:
census_predictors = census_county_year.copy()
census_predictors["acs_source_year"] = census_predictors["year"]
census_predictors["year"] = census_predictors["year"] + 2
census_predictors = census_predictors.rename(columns={
    "pop": "acs_lag2_population",
    "median_income": "acs_lag2_median_income",
    "poverty_rate": "acs_lag2_poverty_rate",
    "pct_under5": "acs_lag2_pct_under5",
    "pct_65plus": "acs_lag2_pct_65plus",
})
census_predictors["acs_lag2_population_100k"] = (
    census_predictors["acs_lag2_population"] / 100_000
)

weather_predictors = weather_county_quarter.copy()
weather_predictors["weather_source_year"] = weather_predictors["year"]
weather_predictors["year"] = weather_predictors["year"] + 1
weather_predictors = weather_predictors.rename(columns={
    "weather_days_observed": "weather_lag4_days_observed",
    "tavg_mean": "weather_lag4_tavg_mean",
    "precip_total": "weather_lag4_precip_total",
    "snowfall_total": "weather_lag4_snowfall_total",
    "hot_days": "weather_lag4_hot_days",
    "freeze_days": "weather_lag4_freeze_days",
    "wet_days": "weather_lag4_wet_days",
})

display(
    census_predictors[
        ["fips", "acs_source_year", "year", "acs_lag2_population"]
    ].head()
)
display(
    weather_predictors[
        [
            "fips",
            "weather_source_year",
            "year",
            "quarter",
            "weather_lag4_tavg_mean",
        ]
    ].head()
)


### 9.3 · Merge prepared tables and add known calendar/geography fields

The facility-demand table remains the base. ACS is a many-quarters-to-one-year join, while weather is a one-to-one facility county-quarter join. Validation checks prevent duplicate joins or accidental row multiplication.


In [ ]:
base_row_count = len(demand_history_panel)

quarterly_panel = demand_history_panel.merge(
    census_predictors,
    on=["fips", "year"],
    how="left",
    validate="many_to_one",
)

quarterly_panel = quarterly_panel.merge(
    weather_predictors,
    on=["fips", "year", "quarter", "quarter_num"],
    how="left",
    validate="one_to_one",
)

if len(quarterly_panel) != base_row_count:
    raise ValueError("The external joins changed the number of facility county-quarter rows.")

quarterly_panel["is_covid"] = quarterly_panel["year"].isin(COVID_YEARS).astype(int)
quarterly_panel["post_covid"] = quarterly_panel["year"].ge(2022).astype(int)

quarter_dummies = pd.get_dummies(
    quarterly_panel["quarter"],
    prefix="quarter",
    dtype=int,
)
for column in ["quarter_Q2", "quarter_Q3", "quarter_Q4"]:
    if column not in quarter_dummies:
        quarter_dummies[column] = 0

quarterly_panel = pd.concat(
    [
        quarterly_panel,
        quarter_dummies[["quarter_Q2", "quarter_Q3", "quarter_Q4"]],
    ],
    axis=1,
)

nyc_fips = {"36005", "36047", "36061", "36081", "36085"}
downstate_fips = nyc_fips | {"36059", "36103", "36119", "36087"}

quarterly_panel["is_nyc"] = quarterly_panel["fips"].isin(nyc_fips).astype(int)
quarterly_panel["is_downstate_non_nyc"] = (
    quarterly_panel["fips"].isin(downstate_fips)
    & ~quarterly_panel["fips"].isin(nyc_fips)
).astype(int)

quarterly_panel["region"] = np.select(
    [
        quarterly_panel["is_nyc"].eq(1),
        quarterly_panel["is_downstate_non_nyc"].eq(1),
    ],
    ["nyc", "downstate_non_nyc"],
    default="upstate",
)

quarterly_panel["facility_ed_encounters_per_100k_proxy"] = np.where(
    quarterly_panel["acs_lag2_population"].gt(0),
    quarterly_panel[TARGET] / quarterly_panel["acs_lag2_population"] * 100_000,
    np.nan,
)

quarterly_panel = (
    quarterly_panel.sort_values(["year", "quarter_num", "fips"])
    .reset_index(drop=True)
)

if quarterly_panel.duplicated(["fips", "year", "quarter"]).any():
    raise ValueError("Duplicate facility county-quarter rows were found.")

quarterly_panel.to_csv(ANALYSIS_TABLE_PATH, index=False)

print("Final facility county-quarter analytical panel:", quarterly_panel.shape)
print("Each row represents one facility county and one calendar quarter.")

display(
    quarterly_panel[
        [
            "county_name",
            "year",
            "quarter",
            TARGET,
            "target_lag1",
            "target_lag4",
            "acs_source_year",
            "weather_source_year",
        ]
    ].head(12)
)


## 10 · Task 2 quality checks, coverage, and holdout year

The holdout is the latest year with four sufficiently complete quarters and available two-year-lag ACS and prior-year weather. “Sufficiently complete” means each quarter contains at least 90% of the largest facility-county count observed in any quarter.

The quality review also checks duplicate facility county-quarter keys, missing targets, source-year alignment, suppressed values, component consistency, and county coverage.


In [ ]:
key_columns = ["fips", "year", "quarter"]

quality_summary = pd.DataFrame({
    "check": [
        "rows",
        "duplicate facility county-quarter keys",
        "facility counties represented",
        "target years",
        "missing target values",
        "suppressed or nonnumeric facility targets",
    ],
    "value": [
        len(quarterly_panel),
        int(quarterly_panel.duplicated(key_columns).sum()),
        int(quarterly_panel["fips"].nunique()),
        f"{quarterly_panel['year'].min()}–{quarterly_panel['year'].max()}",
        int(quarterly_panel[TARGET].isna().sum()),
        SUPPRESSED_TARGET_ROWS,
    ],
})

display(quality_summary)

represented_fips = set(quarterly_panel["fips"].astype(str))
missing_facility_counties = (
    county_geo.loc[
        ~county_geo["fips"].astype(str).isin(represented_fips),
        ["fips", "county_name"],
    ]
    .drop_duplicates()
    .sort_values("county_name")
    .reset_index(drop=True)
)

print("NY counties in the geography crosswalk:", county_geo["fips"].nunique())
print("Facility counties represented:", quarterly_panel["fips"].nunique())
print(
    "Counties without qualifying facility-quarter target rows:",
    len(missing_facility_counties),
)
display(missing_facility_counties)

quarter_coverage = (
    quarterly_panel.groupby(["year", "quarter"])["fips"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["Q1", "Q2", "Q3", "Q4"], fill_value=0)
)
display(quarter_coverage)

largest_quarter_count = int(quarter_coverage.to_numpy().max())
minimum_complete_count = math.ceil(0.90 * largest_quarter_count)

complete_years = quarter_coverage.index[
    quarter_coverage.ge(minimum_complete_count).all(axis=1)
]

predictor_year_limit = min(
    int(census_county_year["year"].max()) + 2,
    int(weather_county_quarter["year"].max()) + 1,
)

eligible_holdout_years = [
    int(year)
    for year in complete_years
    if int(year) <= predictor_year_limit
]

if not eligible_holdout_years:
    raise ValueError("No complete target year has the required lagged predictors.")

HOLDOUT_YEAR = max(eligible_holdout_years)
TRAIN_END_YEAR = HOLDOUT_YEAR - 1

alignment_rows = quarterly_panel.dropna(
    subset=["acs_source_year", "weather_source_year"]
)
if not (
    alignment_rows["acs_source_year"].astype(int)
    .eq(alignment_rows["year"].astype(int) - 2)
    .all()
):
    raise ValueError("ACS source years are not aligned to two years before the target.")

if not (
    alignment_rows["weather_source_year"].astype(int)
    .eq(alignment_rows["year"].astype(int) - 1)
    .all()
):
    raise ValueError("Weather source years are not aligned to the prior target year.")

important_missing = quarterly_panel[
    [
        TARGET,
        "target_lag1",
        "target_lag4",
        "acs_lag2_population",
        "acs_lag2_median_income",
        "weather_lag4_tavg_mean",
        "weather_lag4_precip_total",
    ]
].isna().mean().mul(100).round(2).rename("missing_percent")

display(important_missing.to_frame())

print("Largest county count in one quarter:", largest_quarter_count)
print("Minimum count required for a complete quarter:", minimum_complete_count)
print("Final holdout year:", HOLDOUT_YEAR)
print("Weather-day coverage for aligned quarters:")
display(
    quarterly_panel["weather_lag4_days_observed"]
    .describe()
    .to_frame()
    .T
    .round(1)
)

print(
    "Component-gap is a diagnostic only; suppressed component cells can affect it."
)
display(quarterly_panel["component_gap"].describe().to_frame().T.round(2))


### Task 2 note

The cleaning workflow keeps the facility-demand table as the panel anchor, aggregates only valid facility rows to facility county-quarter, treats unavailable ACS values as missing, and reports public suppression explicitly. Two-year-lag ACS and prior-year weather are aligned before modeling. Facility county is the operational geography and is not necessarily the patient's county of residence.


# Phase 4 — Exploratory Data Analysis & Visualization

## Task 3 — Exploratory Data Analysis & Visualization

The pre-holdout EDA is stakeholder-oriented. It establishes statewide demand and quarterly seasonality, identifies high-volume facility counties, examines demographic and weather relationships, and evaluates correlation and multicollinearity before model development.


## 11 · Statewide context, quarterly seasonality, and county demand

Official statewide rows support the project introduction. County modeling uses actual facility rows aggregated by facility county. Statewide values are not merged into county observations.

The statewide trend and latest-year county totals are descriptive reporting context. They are not used to select features or models. The target distribution, seasonality calculation, demographic/weather relationships, correlation analysis, and feature selection use pre-holdout rows only.


In [ ]:
pre_holdout_eda = quarterly_panel.loc[
    quarterly_panel["year"] < HOLDOUT_YEAR
].copy()

statewide = facility_raw.loc[
    facility_raw["facility_name"].astype(str).str.strip().str.casefold().eq("statewide")
    & facility_raw["quarter"].astype(str).str.upper().isin(["Q1", "Q2", "Q3", "Q4"])
].copy()

statewide["year"] = to_num(statewide["year"])
statewide["quarter_num"] = statewide["quarter"].str[-1].astype(int)
statewide["period"] = pd.PeriodIndex(
    year=statewide["year"].astype(int),
    quarter=statewide["quarter_num"],
    freq="Q",
).to_timestamp()
statewide[TARGET] = to_num(statewide[TARGET])
statewide = statewide.loc[statewide["year"] < HOLDOUT_YEAR].sort_values("period")

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(statewide["period"], statewide[TARGET], marker="o")
ax.axvspan(
    pd.Timestamp("2020-01-01"),
    pd.Timestamp("2021-12-31"),
    alpha=0.12,
    label="COVID period",
)
add_line_value_labels(
    ax,
    statewide["period"],
    statewide[TARGET],
    formatter=format_count,
    fontsize=7,
)
ax.set_title("New York Statewide Quarterly ED Encounters")
ax.set_xlabel("Quarter")
ax.set_ylabel("Total ED encounters")
ax.grid(alpha=0.3)
ax.legend()
ax.margins(y=0.16)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "statewide_trend.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
hist_counts, hist_bins, hist_patches = ax.hist(
    np.log1p(pre_holdout_eda[TARGET].dropna()),
    bins=30,
)
add_histogram_count_labels(ax, hist_patches, fontsize=7)
ax.set_title("Distribution of County-Quarter ED Volume, Pre-Holdout")
ax.set_xlabel("log(1 + total ED encounters)")
ax.set_ylabel("County-quarter rows")
ax.margins(y=0.15)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "target_distribution_log.png", dpi=200, bbox_inches="tight")
plt.show()

seasonality = (
    pre_holdout_eda
    .groupby("quarter_num")[TARGET]
    .median()
    .reindex([1, 2, 3, 4])
)

fig, ax = plt.subplots(figsize=(7, 5))
seasonality_bars = ax.bar(
    ["Q1", "Q2", "Q3", "Q4"],
    seasonality.values,
)
add_bar_value_labels(
    ax,
    seasonality_bars,
    formatter=format_count,
)
ax.set_title("Median County ED Volume by Quarter, Pre-Holdout")
ax.set_xlabel("Quarter")
ax.set_ylabel("Median ED encounters")
ax.margins(y=0.15)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "seasonality.png", dpi=200, bbox_inches="tight")
plt.show()

latest_descriptive_year = int(pre_holdout_eda["year"].max())
top_counties = (
    pre_holdout_eda.loc[pre_holdout_eda["year"].eq(latest_descriptive_year)]
    .groupby("county_name")[TARGET]
    .sum()
    .nlargest(10)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 6))
top_county_bars = ax.barh(top_counties.index, top_counties.values)
add_bar_value_labels(
    ax,
    top_county_bars,
    formatter=format_count,
    horizontal=True,
)
ax.set_title(f"Highest-Volume Facility Counties, {latest_descriptive_year}")
ax.set_xlabel("Total ED encounters across four quarters")
ax.set_ylabel("Facility county")
ax.set_xlim(right=top_counties.max() * 1.17)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "top_counties.png", dpi=200, bbox_inches="tight")
plt.show()


### EDA interpretation note

The pre-holdout statewide series was stable near two million encounters per quarter before the pandemic, fell sharply during the 2020 disruption, and then recovered toward its earlier range. The log-scale histogram is used only to display the large difference between small and large facility counties; the forecasting target remains raw encounter counts.

Median county volume is highest in Q3, but the seasonal difference is modest relative to the much larger differences between counties. The highest-volume chart is based on **facility location** and four-quarter totals, not unique patients or county-of-residence utilization. A county with many major hospitals can therefore have more facility encounters than its resident population alone would suggest.

## 12 · Demographic and weather relationships

Raw encounter volume is appropriate for staffing and capacity planning. The two-year-lag population-standardized value is a descriptive **facility-location proxy**, not a resident utilization rate.

Weather relationships use the change from the same quarter in the previous year. This reduces the dominance of stable county-size differences.


In [ ]:
eda_data = quarterly_panel.loc[
    quarterly_panel["year"] < HOLDOUT_YEAR
].copy()
eda_data["observation_label"] = (
    eda_data["county_name"].astype(str)
    + " — "
    + eda_data["year"].astype(int).astype(str)
    + " "
    + eda_data["quarter"].astype(str)
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(eda_data["acs_lag2_population"], eda_data[TARGET], alpha=0.35)
add_selected_scatter_labels(
    ax,
    eda_data,
    x_column="acs_lag2_population",
    y_column=TARGET,
    label_column="observation_label",
    n_labels=10,
    x_formatter=format_count,
    y_formatter=format_count,
    note="Labels show the 10 highest-volume observations.",
)
ax.set_title("County Population and Quarterly ED Volume, Pre-Holdout")
ax.set_xlabel("Two-year-lag county population")
ax.set_ylabel("Quarterly ED encounters")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "population_vs_volume.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    eda_data["acs_lag2_median_income"],
    eda_data["facility_ed_encounters_per_100k_proxy"],
    alpha=0.35,
)
add_selected_scatter_labels(
    ax,
    eda_data,
    x_column="acs_lag2_median_income",
    y_column="facility_ed_encounters_per_100k_proxy",
    label_column="observation_label",
    n_labels=10,
    x_formatter=lambda value: f"${value:,.0f}",
    y_formatter=format_count,
    note="Labels show the 10 highest ED-rate proxy observations.",
)
ax.set_title("Median Income and Facility-Location ED Proxy, Pre-Holdout")
ax.set_xlabel("Two-year-lag median household income")
ax.set_ylabel("ED encounters per 100,000 population proxy")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "income_vs_ed_proxy.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    eda_data["acs_lag2_poverty_rate"],
    eda_data["facility_ed_encounters_per_100k_proxy"],
    alpha=0.35,
)
add_selected_scatter_labels(
    ax,
    eda_data,
    x_column="acs_lag2_poverty_rate",
    y_column="facility_ed_encounters_per_100k_proxy",
    label_column="observation_label",
    n_labels=10,
    x_formatter=lambda value: f"{value:.1f}%",
    y_formatter=format_count,
    note="Labels show the 10 highest ED-rate proxy observations.",
)
ax.set_title("Poverty and Facility-Location ED Proxy, Pre-Holdout")
ax.set_xlabel("Two-year-lag poverty rate (%)")
ax.set_ylabel("ED encounters per 100,000 population proxy")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "poverty_vs_ed_proxy.png", dpi=200, bbox_inches="tight")
plt.show()

eda_data["year_over_year_change"] = eda_data[TARGET] - eda_data["target_lag4"]

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    eda_data["weather_lag4_tavg_mean"],
    eda_data["year_over_year_change"],
    alpha=0.35,
)
ax.axhline(0, linestyle="--", linewidth=1)
add_selected_scatter_labels(
    ax,
    eda_data,
    x_column="weather_lag4_tavg_mean",
    y_column="year_over_year_change",
    label_column="observation_label",
    score_column="year_over_year_change",
    n_labels=10,
    x_formatter=lambda value: f"{value:.1f}°F",
    y_formatter=lambda value: f"{value:+,.0f}",
    note="Labels show the 10 largest absolute year-over-year changes.",
)
ax.set_title("Prior-Year Temperature and Year-over-Year ED Change, Pre-Holdout")
ax.set_xlabel("Same-quarter previous-year average temperature (°F)")
ax.set_ylabel("Change from same quarter previous year")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "temperature_vs_ed_change.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    eda_data["weather_lag4_precip_total"],
    eda_data["year_over_year_change"],
    alpha=0.35,
)
ax.axhline(0, linestyle="--", linewidth=1)
add_selected_scatter_labels(
    ax,
    eda_data,
    x_column="weather_lag4_precip_total",
    y_column="year_over_year_change",
    label_column="observation_label",
    score_column="year_over_year_change",
    n_labels=10,
    x_formatter=lambda value: f"{value:.1f} in",
    y_formatter=lambda value: f"{value:+,.0f}",
    note="Labels show the 10 largest absolute year-over-year changes.",
)
ax.set_title("Prior-Year Precipitation and Year-over-Year ED Change, Pre-Holdout")
ax.set_xlabel("Same-quarter previous-year precipitation (inches)")
ax.set_ylabel("Change from same quarter previous year")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMAGE_DIR / "precipitation_vs_ed_change.png", dpi=200, bbox_inches="tight")
plt.show()

relationship_columns = [
    "facility_ed_encounters_per_100k_proxy",
    "acs_lag2_median_income",
    "acs_lag2_poverty_rate",
    "acs_lag2_pct_under5",
    "acs_lag2_pct_65plus",
    "year_over_year_change",
    "weather_lag4_tavg_mean",
    "weather_lag4_precip_total",
    "weather_lag4_snowfall_total",
]

relationship_table = (
    eda_data[relationship_columns]
    .corr(method="spearman")
    .round(3)
)

display(relationship_table)
relationship_table.to_csv(PROCESSED_DIR / "eda_spearman_correlations.csv")


### Demographic and weather interpretation note

Two-year-lag county population has a strong positive relationship with raw quarterly encounter volume, as expected for an operational demand target. New York County sits above similarly populated counties because the numerator follows facility location and includes patients who may live elsewhere.

The income and poverty plots do not show a clean linear relationship. Montgomery County repeatedly appears among the largest per-capita proxy values, but this proxy combines facility-location encounters with resident population and must not be interpreted as a resident ED-use rate or as evidence that income or poverty causes ED demand.

The temperature and precipitation plots show no clear monotonic relationship with year-over-year encounter change. The largest changes are concentrated in large New York City facility counties and pandemic/recovery periods, indicating that broad time shocks can dominate weather in these descriptive plots. Weather value should therefore be judged through out-of-time validation and SHAP, not from these scatterplots alone.

## 13 · Correlation and multicollinearity

The heatmap is descriptive. VIF is used to identify linear redundancy before modeling. `target_roll4` is retained in the analytical table for interpretation but excluded from the candidate feature set because it is an exact average of `target_lag1` through `target_lag4`.

**Task 3 deliverable:** labeled visualizations and written interpretation covering quarterly seasonality, geographic variation, volume-versus-population caveats, predictor relationships, and potential leakage or redundancy.


In [ ]:
correlation_columns = [
    TARGET,
    "target_lag1",
    "target_lag4",
    "target_roll4",
    "facility_count_lag1",
    "acs_lag2_population_100k",
    "acs_lag2_median_income",
    "acs_lag2_poverty_rate",
    "acs_lag2_pct_under5",
    "acs_lag2_pct_65plus",
    "weather_lag4_tavg_mean",
    "weather_lag4_precip_total",
    "weather_lag4_snowfall_total",
]

correlation_matrix = eda_data[correlation_columns].corr(method="spearman")

fig, ax = plt.subplots(figsize=(13, 10))
correlation_image = ax.imshow(
    correlation_matrix,
    aspect="auto",
    vmin=-1,
    vmax=1,
)
fig.colorbar(correlation_image, ax=ax, label="Spearman correlation")
ax.set_xticks(range(len(correlation_columns)))
ax.set_xticklabels(
    correlation_columns,
    rotation=75,
    ha="right",
)
ax.set_yticks(range(len(correlation_columns)))
ax.set_yticklabels(correlation_columns)

for row_index in range(correlation_matrix.shape[0]):
    for column_index in range(correlation_matrix.shape[1]):
        value = correlation_matrix.iloc[row_index, column_index]
        if pd.isna(value):
            continue
        ax.text(
            column_index,
            row_index,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            color="white" if abs(value) >= 0.55 else "black",
        )

ax.set_title("Correlation Matrix of Quarterly ED Features")
fig.tight_layout()
fig.savefig(IMAGE_DIR / "correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()


def calculate_vif(frame, columns):
    complete = frame[columns].dropna().astype(float)
    rows = []

    for feature in columns:
        other_features = [column for column in columns if column != feature]
        model = LinearRegression().fit(
            complete[other_features],
            complete[feature],
        )
        r_squared = model.score(
            complete[other_features],
            complete[feature],
        )
        vif = np.inf if r_squared >= 0.999999 else 1 / (1 - r_squared)
        rows.append({"feature": feature, "VIF": vif})

    return pd.DataFrame(rows).sort_values("VIF", ascending=False)


vif_columns = [
    "target_lag1",
    "target_lag2",
    "target_lag3",
    "target_lag4",
    "facility_count_lag1",
    "acs_lag2_population_100k",
    "acs_lag2_median_income",
    "acs_lag2_poverty_rate",
    "acs_lag2_pct_under5",
    "acs_lag2_pct_65plus",
    "weather_lag4_tavg_mean",
    "weather_lag4_precip_total",
    "weather_lag4_snowfall_total",
]

vif_table = calculate_vif(eda_data, vif_columns)
display(vif_table.round(2))
vif_table.to_csv(PROCESSED_DIR / "multicollinearity_vif.csv", index=False)


### Correlation interpretation note

The target is strongly associated with recent demand history and county population. This is expected: county scale is persistent, and adjacent quarterly volumes are highly related. These correlations are descriptive rather than causal.

The very strong negative relationship between temperature and snowfall indicates redundant seasonal information. High correlations among `target_lag1`, `target_lag4`, and `target_roll4` also mean that feature importance can be shared across related variables. `target_roll4` is excluded because it is constructed directly from the four lag variables; remaining predictors are handled through fold-specific selection and regularization rather than being removed solely because of pairwise correlation.

# Phase 5 — Feature Engineering, Selection, and Modeling

## Task 4 — Feature Engineering & Selection

The candidate pool combines demand-history, demographic, weather, calendar, COVID-period, facility-history, and geographic predictors. Feature definitions remain quarterly, and all selection steps are fitted only on the relevant training period.


## 14 · Task 4 leakage-safe predictor set

Excluded from modeling:

- current-quarter weather;
- current-quarter encounter components and current-quarter facility count;
- numeric year;
- the descriptive per-capita proxy;
- `target_roll4`, because it is an exact linear combination of the four lag columns.

The two geography indicators use upstate as the reference group, so no redundant third region dummy is added. Tree models do not extrapolate a numeric time trend reliably; demand history carries the level, while quarter and COVID flags are retained only when selected from training data.


In [ ]:
DEMOGRAPHIC_FEATURES = [
    "acs_lag2_population_100k",
    "acs_lag2_median_income",
    "acs_lag2_poverty_rate",
    "acs_lag2_pct_under5",
    "acs_lag2_pct_65plus",
]

WEATHER_FEATURES = [
    "weather_lag4_tavg_mean",
    "weather_lag4_precip_total",
    "weather_lag4_snowfall_total",
    "weather_lag4_hot_days",
    "weather_lag4_freeze_days",
    "weather_lag4_wet_days",
]

HISTORY_FEATURES = [
    "target_lag1",
    "target_lag2",
    "target_lag3",
    "target_lag4",
    "facility_count_lag1",
]

CALENDAR_FEATURES = [
    "quarter_Q2",
    "quarter_Q3",
    "quarter_Q4",
    "is_covid",
    "post_covid",
]

GEOGRAPHY_FEATURES = [
    "lat",
    "lon",
    "is_nyc",
    "is_downstate_non_nyc",
]

FEATURE_COLUMNS = (
    DEMOGRAPHIC_FEATURES
    + WEATHER_FEATURES
    + HISTORY_FEATURES
    + CALENDAR_FEATURES
    + GEOGRAPHY_FEATURES
)

FEATURE_GROUPS = {
    **{feature: "Demographics" for feature in DEMOGRAPHIC_FEATURES},
    **{feature: "Prior-year weather" for feature in WEATHER_FEATURES},
    **{feature: "Demand history" for feature in HISTORY_FEATURES},
    **{feature: "Calendar" for feature in CALENDAR_FEATURES},
    **{feature: "Geography" for feature in GEOGRAPHY_FEATURES},
}

feature_dictionary = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "group": [FEATURE_GROUPS[feature] for feature in FEATURE_COLUMNS],
})

display(feature_dictionary)
feature_dictionary.to_csv(REPORT_DIR / "feature_dictionary.csv", index=False)

print("Candidate predictors:", len(FEATURE_COLUMNS))


## 15 · Task 4 final holdout and model-ready rows

The final holdout is one complete calendar year containing four sequential one-quarter-ahead evaluation quarters. Model selection and tuning use only earlier quarters.


In [ ]:
model_ready = quarterly_panel.dropna(
    subset=FEATURE_COLUMNS + [TARGET]
).copy()

holdout_train = model_ready.loc[
    model_ready["year"] <= TRAIN_END_YEAR
].copy()

holdout_test = model_ready.loc[
    model_ready["year"] == HOLDOUT_YEAR
].copy()

if holdout_train.empty or holdout_test.empty:
    raise ValueError("The model-ready training or holdout table is empty.")

holdout_quarters = sorted(holdout_test["quarter"].unique())
if holdout_quarters != ["Q1", "Q2", "Q3", "Q4"]:
    raise ValueError(
        f"The final holdout does not contain all four quarters: {holdout_quarters}"
    )

holdout_model_coverage = (
    holdout_test.groupby("quarter")["fips"]
    .nunique()
    .reindex(["Q1", "Q2", "Q3", "Q4"], fill_value=0)
)
if holdout_model_coverage.lt(minimum_complete_count).any():
    raise ValueError(
        "The model-ready holdout has insufficient county coverage: "
        f"{holdout_model_coverage.to_dict()}"
    )

print(
    "Training years:",
    int(holdout_train["year"].min()),
    "to",
    TRAIN_END_YEAR,
)
print("Final holdout year:", HOLDOUT_YEAR)
print("Training rows:", len(holdout_train))
print("Holdout rows:", len(holdout_test))
print("Holdout quarters:", holdout_quarters)
print("Model-ready holdout county coverage:")
display(holdout_model_coverage.to_frame("counties"))


## 16 · Task 4 visible, leakage-safe feature selection

`SelectKBest` with F-regression is used as a transparent screening step. It is fitted on the training period only and refitted inside every rolling fold.

Separate feature lists are learned for the level target and the change-from-seasonal-persistence target. Forward and backward stepwise selection are not used because the lag predictors are correlated, stepwise subsets are unstable across time, and stepwise procedures are not naturally aligned with the nonlinear candidate models.


In [ ]:
DEFAULT_FEATURE_SELECTION_K = 15
FEATURE_SELECTION_K_OPTIONS = [10, 15, 20]


def target_for_mode(frame, target_mode):
    if target_mode == "change":
        # Quarterly equivalent of "change from last year."
        return frame[TARGET] - frame["target_lag4"]
    return frame[TARGET]


def select_features(
    train_frame,
    target_mode,
    k=DEFAULT_FEATURE_SELECTION_K,
):
    usable_features = [
        feature
        for feature in FEATURE_COLUMNS
        if train_frame[feature].nunique(dropna=True) > 1
    ]

    if not usable_features:
        raise ValueError("No non-constant candidate features are available.")

    selector = SelectKBest(
        score_func=f_regression,
        k=min(int(k), len(usable_features)),
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        selector.fit(
            train_frame[usable_features],
            target_for_mode(train_frame, target_mode),
        )

    report = pd.DataFrame({
        "feature": usable_features,
        "feature_group": [
            FEATURE_GROUPS[feature]
            for feature in usable_features
        ],
        "f_score": selector.scores_,
        "p_value": selector.pvalues_,
        "selected": selector.get_support(),
    })

    report["f_score"] = report["f_score"].replace(
        [np.inf, -np.inf],
        np.nan,
    )
    report = report.sort_values(
        ["selected", "f_score"],
        ascending=[False, False],
    ).reset_index(drop=True)
    report["rank"] = np.arange(1, len(report) + 1)

    selected_features = report.loc[
        report["selected"],
        "feature",
    ].tolist()

    return selected_features, selector, report


level_features, _, level_feature_report = select_features(
    holdout_train,
    "level",
)
change_features, _, change_feature_report = select_features(
    holdout_train,
    "change",
)

print("Default feature count:", DEFAULT_FEATURE_SELECTION_K)
print("Feature counts considered during tuning:", FEATURE_SELECTION_K_OPTIONS)
print("Level-target feature screening:")
display(level_feature_report.head(20).round(4))

print("Change-from-seasonal-persistence feature screening:")
display(change_feature_report.head(20).round(4))

level_feature_report.to_csv(
    PROCESSED_DIR / "feature_selection_level.csv",
    index=False,
)
change_feature_report.to_csv(
    PROCESSED_DIR / "feature_selection_change.csv",
    index=False,
)


### Task 4 note

Feature selection is target-specific and leakage-safe. The fixed holdout feature lists are learned from pre-holdout rows only. During rolling-origin validation, the selector is refitted inside each fold using only quarters earlier than the validation quarter, and fold-constant predictors are removed before F-scoring.


## Week of July 14 — Model Selection and Development

### Task 5 — Persistence baselines, metrics, and compact candidate set

The main metric is MAE. Skill is the fraction of persistence MAE removed:

`skill = 1 - model_MAE / benchmark_MAE`

Positive skill means the model beats the benchmark. WAPE expresses total absolute error as a percentage of total observed encounter volume, so lower values are better. Pooled R² is retained but treated as secondary because county volumes differ greatly in scale.

After discussion with the professor, XGBoost replaces Hist Gradient Boosting. Each retained model family is evaluated under both target formulations so the comparison is balanced:

- Ridge — level;
- Ridge — change from seasonal persistence;
- Random Forest — level;
- Random Forest — change from seasonal persistence;
- XGBoost — level;
- XGBoost — change from seasonal persistence.

For the change formulation, the target is current encounters minus `target_lag4`. The predicted change is added back to `target_lag4` to reconstruct the encounter level. This directly tests whether two-year-lag demographics, prior-year weather, geography, and recent demand history improve on carrying the same quarter from the previous year forward.

**Week of July 14 deliverable:** define common evaluation functions, compare all six candidates against the quarterly persistence baselines, and carry the best-supported model family and target formulation into rolling validation.


In [ ]:
def regression_metrics(actual, prediction):
    actual = np.asarray(actual)
    prediction = np.asarray(prediction)

    mae = mean_absolute_error(actual, prediction)
    rmse = mean_squared_error(actual, prediction) ** 0.5
    wape = np.abs(actual - prediction).sum() / np.abs(actual).sum()

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape,
        "R2": r2_score(actual, prediction),
    }


def skill_vs(actual, prediction, benchmark):
    model_mae = mean_absolute_error(actual, prediction)
    benchmark_mae = mean_absolute_error(actual, benchmark)

    if benchmark_mae == 0:
        return np.nan

    return 1 - model_mae / benchmark_mae


def change_r2(actual, prediction, reference):
    actual_change = np.asarray(actual) - np.asarray(reference)
    predicted_change = np.asarray(prediction) - np.asarray(reference)

    if np.allclose(actual_change, actual_change[0]):
        return np.nan

    return r2_score(actual_change, predicted_change)


RANDOM_FOREST_BASE = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1,
)

XGBOOST_BASE = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="mae",
    tree_method="hist",
    n_estimators=350,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

CANDIDATE_SPECS = {
    "Ridge — level": {
        "family": "Ridge",
        "target_mode": "level",
        "needs_scaling": True,
        "estimator": Ridge(alpha=1.0),
    },
    "Ridge — change from seasonal persistence": {
        "family": "Ridge",
        "target_mode": "change",
        "needs_scaling": True,
        "estimator": Ridge(alpha=1.0),
    },
    "Random Forest — level": {
        "family": "Random Forest",
        "target_mode": "level",
        "needs_scaling": False,
        "estimator": RANDOM_FOREST_BASE,
    },
    "Random Forest — change from seasonal persistence": {
        "family": "Random Forest",
        "target_mode": "change",
        "needs_scaling": False,
        "estimator": RANDOM_FOREST_BASE,
    },
    "XGBoost — level": {
        "family": "XGBoost",
        "target_mode": "level",
        "needs_scaling": False,
        "estimator": XGBOOST_BASE,
    },
    "XGBoost — change from seasonal persistence": {
        "family": "XGBoost",
        "target_mode": "change",
        "needs_scaling": False,
        "estimator": XGBOOST_BASE,
    },
}


def fit_predict(train_frame, test_frame, spec, features, estimator=None):
    model = clone(
        estimator if estimator is not None
        else spec["estimator"]
    )

    X_train = train_frame[features]
    X_test = test_frame[features]

    scaler = None
    if spec["needs_scaling"]:
        scaler = StandardScaler().fit(X_train)
        X_train_fit = scaler.transform(X_train)
        X_test_fit = scaler.transform(X_test)
    else:
        X_train_fit = X_train
        X_test_fit = X_test

    model.fit(
        X_train_fit,
        target_for_mode(train_frame, spec["target_mode"]),
    )
    component_prediction = model.predict(X_test_fit)

    if spec["target_mode"] == "change":
        level_prediction = (
            test_frame["target_lag4"].to_numpy()
            + component_prediction
        )
    else:
        level_prediction = component_prediction

    return {
        "model": model,
        "scaler": scaler,
        "prediction": np.asarray(level_prediction),
    }


### Week of July 14 deliverable — candidate model set

Ridge, Random Forest, and XGBoost are evaluated under the same complete-quarter rolling-origin design. The machine-learning specification with the lowest mean rolling MAE is carried forward to tuning. The exact winner and comparison paragraph are generated directly from the results in Task 6 so the narrative stays synchronized after rerunning the notebook.


## Week of July 21 — Model Evaluation and Validation

### Task 6 — Explicit rolling-origin validation by complete quarter

A row-based `TimeSeriesSplit` is not appropriate for a county-quarter panel because rows from the same calendar quarter could appear on both sides of a split.

Each expanding-window fold:

- trains on every earlier complete quarter;
- validates on one later complete quarter across counties;
- refits feature selection, scaling, and the model inside the fold;
- reports MAE, RMSE, WAPE, and R² for every model and persistence benchmark;
- compares MAE with both previous-quarter and same-quarter-last-year persistence;
- uses the strongest persistence benchmark for the reported skill score;
- uses mean rolling MAE as the primary model-selection criterion, with R² retained as a secondary diagnostic;
- summarizes stability by validation quarter, calendar quarter, region, and county.

The first eight model-ready quarters establish the initial training history. This section is the formal **through-July-21 validation checkpoint**.


In [ ]:
pre_holdout = model_ready.loc[
    model_ready["year"] < HOLDOUT_YEAR
].copy()

all_pre_holdout_periods = sorted(
    pre_holdout["period_index"].unique()
)
validation_periods = all_pre_holdout_periods[
    MIN_TRAIN_QUARTERS:
]

if not validation_periods:
    raise ValueError("There are no rolling validation quarters.")

rolling_metric_rows = []
rolling_prediction_rows = []
selection_rows = []

for validation_period in validation_periods:
    fold_train = pre_holdout.loc[
        pre_holdout["period_index"] < validation_period
    ].copy()

    fold_test = pre_holdout.loc[
        pre_holdout["period_index"] == validation_period
    ].copy()

    actual = fold_test[TARGET].to_numpy()
    baseline_predictions = {
        "Previous-quarter persistence": fold_test["target_lag1"].to_numpy(),
        "Seasonal persistence": fold_test["target_lag4"].to_numpy(),
    }

    for model_name, prediction in baseline_predictions.items():
        rolling_metric_rows.append({
            "validation_period": validation_period,
            "year": int(fold_test["year"].iloc[0]),
            "quarter": fold_test["quarter"].iloc[0],
            "model": model_name,
            **regression_metrics(actual, prediction),
        })

        for row_index, predicted_value in zip(fold_test.index, prediction):
            rolling_prediction_rows.append({
                "validation_period": validation_period,
                "year": int(fold_test.loc[row_index, "year"]),
                "quarter": fold_test.loc[row_index, "quarter"],
                "fips": fold_test.loc[row_index, "fips"],
                "county_name": fold_test.loc[row_index, "county_name"],
                "region": fold_test.loc[row_index, "region"],
                "model": model_name,
                "actual": float(fold_test.loc[row_index, TARGET]),
                "prediction": float(predicted_value),
            })

    for model_name, spec in CANDIDATE_SPECS.items():
        fold_features, _, _ = select_features(
            fold_train,
            spec["target_mode"],
        )

        fitted = fit_predict(
            fold_train,
            fold_test,
            spec,
            fold_features,
        )
        prediction = fitted["prediction"]

        rolling_metric_rows.append({
            "validation_period": validation_period,
            "year": int(fold_test["year"].iloc[0]),
            "quarter": fold_test["quarter"].iloc[0],
            "model": model_name,
            **regression_metrics(actual, prediction),
        })

        for feature in fold_features:
            selection_rows.append({
                "validation_period": validation_period,
                "model": model_name,
                "feature": feature,
            })

        for row_index, predicted_value in zip(fold_test.index, prediction):
            rolling_prediction_rows.append({
                "validation_period": validation_period,
                "year": int(fold_test.loc[row_index, "year"]),
                "quarter": fold_test.loc[row_index, "quarter"],
                "fips": fold_test.loc[row_index, "fips"],
                "county_name": fold_test.loc[row_index, "county_name"],
                "region": fold_test.loc[row_index, "region"],
                "model": model_name,
                "actual": float(fold_test.loc[row_index, TARGET]),
                "prediction": float(predicted_value),
            })

rolling_results = pd.DataFrame(rolling_metric_rows)
rolling_predictions = pd.DataFrame(rolling_prediction_rows)
rolling_predictions["residual"] = (
    rolling_predictions["actual"]
    - rolling_predictions["prediction"]
)
rolling_predictions["absolute_error"] = (
    rolling_predictions["residual"].abs()
)

benchmark_names = [
    "Previous-quarter persistence",
    "Seasonal persistence",
]

benchmark_summary = (
    rolling_results.loc[
        rolling_results["model"].isin(benchmark_names)
    ]
    .groupby("model", as_index=False)["MAE"]
    .mean()
    .sort_values("MAE")
)

STRONGEST_BENCHMARK = benchmark_summary.iloc[0]["model"]

benchmark_by_period = (
    rolling_results.loc[
        rolling_results["model"].eq(STRONGEST_BENCHMARK),
        ["validation_period", "MAE"],
    ]
    .rename(columns={"MAE": "benchmark_MAE"})
)

rolling_results = rolling_results.merge(
    benchmark_by_period,
    on="validation_period",
    how="left",
    validate="many_to_one",
)
rolling_results["skill_vs_strongest_persistence"] = np.where(
    rolling_results["benchmark_MAE"].gt(0),
    1 - rolling_results["MAE"] / rolling_results["benchmark_MAE"],
    np.nan,
)
rolling_results.loc[
    rolling_results["model"].eq(STRONGEST_BENCHMARK),
    "skill_vs_strongest_persistence",
] = 0.0

rolling_summary = (
    rolling_results.groupby("model", as_index=False)
    .agg(
        mean_MAE=("MAE", "mean"),
        median_MAE=("MAE", "median"),
        mean_RMSE=("RMSE", "mean"),
        mean_WAPE=("WAPE", "mean"),
        mean_R2=("R2", "mean"),
        median_R2=("R2", "median"),
        mean_skill=("skill_vs_strongest_persistence", "mean"),
        quarters_beating_benchmark=(
            "skill_vs_strongest_persistence",
            lambda values: int((values > 0).sum()),
        ),
        validation_quarters=("validation_period", "nunique"),
    )
    .sort_values("mean_MAE")
    .reset_index(drop=True)
)

ml_names = list(CANDIDATE_SPECS)
WINNING_ML_NAME = (
    rolling_summary.loc[
        rolling_summary["model"].isin(ml_names)
    ]
    .sort_values("mean_MAE")
    .iloc[0]["model"]
)
WINNING_SPEC = CANDIDATE_SPECS[WINNING_ML_NAME]

display(rolling_summary.round(4))
print("Selected ML candidate:", WINNING_ML_NAME)
print("Strongest persistence benchmark:", STRONGEST_BENCHMARK)

winning_rolling_row = rolling_summary.loc[
    rolling_summary["model"].eq(WINNING_ML_NAME)
].iloc[0]

rolling_verdict = (
    f"{WINNING_ML_NAME} had mean rolling MAE "
    f"{winning_rolling_row['mean_MAE']:,.1f}, mean rolling R² "
    f"{winning_rolling_row['mean_R2']:.3f}, and mean skill "
    f"{winning_rolling_row['mean_skill']:.3f} versus "
    f"{STRONGEST_BENCHMARK}. It beat the benchmark in "
    f"{int(winning_rolling_row['quarters_beating_benchmark'])} of "
    f"{int(winning_rolling_row['validation_quarters'])} validation quarters."
)
print(rolling_verdict)

selection_history = pd.DataFrame(selection_rows)
winning_selection_stability = (
    selection_history.loc[
        selection_history["model"].eq(WINNING_ML_NAME)
    ]
    .groupby("feature", as_index=False)
    .agg(selected_quarters=("validation_period", "nunique"))
)
winning_selection_stability["selection_rate"] = (
    winning_selection_stability["selected_quarters"]
    / len(validation_periods)
)
winning_selection_stability = winning_selection_stability.sort_values(
    ["selection_rate", "feature"],
    ascending=[False, True],
)
print("Feature-selection stability for the selected ML candidate:")
display(winning_selection_stability.head(20).round(3))

region_comparison = (
    rolling_predictions.loc[
        rolling_predictions["model"].isin(
            [STRONGEST_BENCHMARK, WINNING_ML_NAME]
        )
    ]
    .groupby(["model", "region"], as_index=False)
    .agg(
        MAE=("absolute_error", "mean"),
        mean_residual=("residual", "mean"),
        rows=("absolute_error", "size"),
    )
)

print("Pre-holdout error by region:")
display(region_comparison.round(2))

rolling_results.to_csv(
    PROCESSED_DIR / "rolling_origin_results.csv",
    index=False,
)
rolling_predictions.to_csv(
    PROCESSED_DIR / "rolling_origin_predictions.csv",
    index=False,
)
rolling_summary.to_csv(
    PROCESSED_DIR / "rolling_origin_summary.csv",
    index=False,
)
winning_selection_stability.to_csv(
    PROCESSED_DIR / "feature_selection_stability.csv",
    index=False,
)
region_comparison.to_csv(
    PROCESSED_DIR / "rolling_error_by_region.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(10, 6))
plot_table = rolling_summary.sort_values("mean_MAE")
rolling_mae_bars = ax.barh(plot_table["model"], plot_table["mean_MAE"])
add_bar_value_labels(
    ax,
    rolling_mae_bars,
    formatter=lambda value: format_decimal(value, 1),
    horizontal=True,
)
ax.invert_yaxis()
ax.set_title("Pre-Holdout Rolling Validation MAE")
ax.set_xlabel("Mean MAE across validation quarters")
ax.set_ylabel("Method")
ax.set_xlim(right=plot_table["mean_MAE"].max() * 1.18)
fig.tight_layout()
fig.savefig(
    IMAGE_DIR / "rolling_validation_mae.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

winning_skill = rolling_results.loc[
    rolling_results["model"].eq(WINNING_ML_NAME)
].sort_values("validation_period")

winning_skill["period_label"] = (
    winning_skill["year"].astype(str)
    + " "
    + winning_skill["quarter"].astype(str)
)

# Required quarterly adaptation of the professor's
# rolling-origin skill-by-year deliverable.
skill_by_quarter_table = winning_skill[
    [
        "year",
        "quarter",
        "MAE",
        "benchmark_MAE",
        "RMSE",
        "WAPE",
        "R2",
        "skill_vs_strongest_persistence",
    ]
].copy()

skill_by_quarter_table = skill_by_quarter_table.rename(
    columns={
        "MAE": "model_MAE",
        "benchmark_MAE": "persistence_MAE",
        "R2": "model_R2",
        "skill_vs_strongest_persistence": "skill",
    }
)

skill_by_quarter_table["WAPE_percent"] = (
    skill_by_quarter_table["WAPE"] * 100
)
skill_by_quarter_table = skill_by_quarter_table[
    [
        "year",
        "quarter",
        "model_MAE",
        "persistence_MAE",
        "RMSE",
        "WAPE_percent",
        "model_R2",
        "skill",
    ]
]

print(f"Rolling-origin results for {WINNING_ML_NAME}:")
display(
    skill_by_quarter_table.round(
        {
            "model_MAE": 1,
            "persistence_MAE": 1,
            "RMSE": 1,
            "WAPE_percent": 2,
            "model_R2": 4,
            "skill": 4,
        }
    )
)

skill_by_quarter_table.to_csv(
    PROCESSED_DIR / "winning_model_skill_by_quarter.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    winning_skill["period_label"],
    winning_skill["skill_vs_strongest_persistence"],
    marker="o",
)
ax.axhline(0, linestyle="--", linewidth=1)
add_line_value_labels(
    ax,
    winning_skill["period_label"],
    winning_skill["skill_vs_strongest_persistence"],
    formatter=lambda value: format_decimal(value, 3),
    fontsize=7,
    offset_points=6,
)
ax.tick_params(axis="x", rotation=75)
ax.set_title(f"Quarterly Skill: {WINNING_ML_NAME}")
ax.set_xlabel("Validation quarter")
ax.set_ylabel(f"Skill vs {STRONGEST_BENCHMARK}")
ax.margins(y=0.18)
fig.tight_layout()
fig.savefig(
    IMAGE_DIR / "rolling_skill_by_quarter.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

# Generate the two required narrative paragraphs directly from the
# rolling-origin results so names, errors, skill, and quarters stay synchronized.
ml_ranking = (
    rolling_summary.loc[rolling_summary["model"].isin(CANDIDATE_SPECS)]
    .sort_values("mean_MAE")
    .reset_index(drop=True)
)
winner_row = ml_ranking.iloc[0]
runner_up_row = ml_ranking.iloc[1] if len(ml_ranking) > 1 else None
benchmark_row_rolling = rolling_summary.loc[
    rolling_summary["model"].eq(STRONGEST_BENCHMARK)
].iloc[0]

runner_up_clause = ""
if runner_up_row is not None:
    runner_up_clause = (
        f", compared with {runner_up_row['mean_MAE']:,.1f} for "
        f"{runner_up_row['model']}"
    )

carry_forward_paragraph = (
    f"{winner_row['model']} is the machine-learning specification carried "
    f"forward to tuning because it achieved the lowest mean rolling-origin "
    f"MAE among the ML candidates at {winner_row['mean_MAE']:,.1f} encounters"
    f"{runner_up_clause}. This selection identifies the strongest ML candidate "
    f"under the common validation design; it does not imply that ML beat "
    f"persistence overall."
)

winning_quarters = rolling_results.loc[
    rolling_results["model"].eq(WINNING_ML_NAME)
    & rolling_results["skill_vs_strongest_persistence"].gt(0),
    ["year", "quarter", "skill_vs_strongest_persistence"],
].sort_values(["year", "quarter"])

if winning_quarters.empty:
    quarter_clause = "It did not beat persistence in any validation quarter."
else:
    quarter_labels = [
        f"{int(row.year)} {row.quarter} ({row.skill_vs_strongest_persistence:.3f} skill)"
        for row in winning_quarters.itertuples()
    ]
    quarter_clause = (
        f"It beat persistence in {len(winning_quarters)} of "
        f"{int(winner_row['validation_quarters'])} quarters: "
        + ", ".join(quarter_labels)
        + "."
    )

rolling_value_paragraph = (
    f"{WINNING_ML_NAME} had mean rolling MAE "
    f"{winner_row['mean_MAE']:,.1f}, compared with "
    f"{benchmark_row_rolling['mean_MAE']:,.1f} for "
    f"{STRONGEST_BENCHMARK}. {quarter_clause} "
    f"Therefore, {STRONGEST_BENCHMARK} remains the stronger overall "
    f"forecasting method when its mean MAE is lower."
)

display(
    Markdown(
        "### Generated weekly verdicts\n\n"
        + carry_forward_paragraph
        + "\n\n"
        + rolling_value_paragraph
    )
)


### Rolling-validation chart interpretation

The MAE chart ranks methods by average out-of-time error; shorter bars are better. Previous-quarter persistence is the strongest overall method. XGBoost — level has the lowest mean rolling-origin MAE among the machine-learning candidates at 2,989.1 encounters, narrowly ahead of Random Forest — level at 2,992.7. Their average performance is therefore practically similar. The level models outperform their corresponding change-from-seasonal-persistence versions, indicating that the available predictors explain encounter levels more reliably than year-over-year deviations.

In the skill chart, zero means equal performance to the strongest persistence benchmark, positive values mean improvement, and negative values mean worse performance. XGBoost — level beats persistence in only 2 of 12 validation quarters. Extremely negative skill can occur when persistence has unusually low error in a quarter, so skill should always be interpreted together with the underlying MAE.

### Week of July 21 deliverable — when the model adds value over persistence

The two verdict paragraphs printed above are generated directly from the rolling-origin tables. They report the selected ML specification, its mean MAE, the strongest persistence benchmark, and the exact quarters in which ML achieved positive skill.


### Progress checkpoint — scope through July 21

- [x] **Phase 1: Configuration** — quarterly target, geography, lags, external-predictor timing, and holdout rules defined
- [x] **Task 1: Data Collection** — SPARCS, county geography, ACS, and weather sources acquired and documented
- [x] **Task 2: Cleaning & Preprocessing** — county-quarter aggregation, type conversion, source alignment, key checks, and coverage review
- [x] **Task 3: EDA & Visualization** — statewide context, seasonality, county demand, demographic/weather relationships, correlation, and VIF
- [x] **Task 4: Feature Engineering & Selection** — exact quarterly lags, leakage exclusions, and target-specific training-only selection
- [x] **Week of July 14 — Task 5: Model Development** — quarterly persistence baselines, compact candidate set, and a paragraph naming the ML family carried forward
- [x] **Week of July 21 — Task 6: Validation** — expanding-window validation by complete quarter, skill versus persistence, and a paragraph explaining when the ML model adds value
- [ ] **Week of July 28 — Tasks 7–10** — tuning, final holdout, SHAP, serialization, inference validation, model card, and results summary


### Week of July 21 summary

1. Retained the county-quarter target and quarterly lag logic (`target_lag1` and `target_lag4`).
2. Preserved two-year-lag ACS and same-quarter previous-year weather alignment.
3. Organized the notebook into explicit phases and Tasks 1–6 through the July 21 checkpoint.
4. Kept the compact candidate set and both level and change-from-seasonal-persistence formulations.
5. Used complete-quarter rolling-origin folds rather than row-based time splits.
6. Refit feature selection, scaling, and model estimation inside every validation fold.
7. Compared candidates with the strongest quarterly persistence benchmark and summarized stability across time and geography.
8. Identified the ML family and target formulation eligible to continue into the July 28 optimization and finalization phase.


# Phase 6 — Model Optimization, Final Evaluation, and Delivery

## Week of July 28 — Tuning, Artifact, and Final Reporting

### Task 7 — Feature pruning and time-aware tuning

Only the selected machine-learning candidate—its family and its target formulation—is tuned. A short, understandable parameter list is evaluated on the last eight pre-holdout validation quarters.

Feature pruning remains leakage-safe:

- fold-constant predictors are removed;
- feature selection is refitted inside each tuning fold;
- feature counts of 10, 15, and 20 are explicitly compared;
- redundant or weak predictors are not carried into the final artifact merely because they were available;
- every tuning configuration is compared with both the untuned candidate and the strongest persistence benchmark on the same quarters;
- the final holdout remains locked and unused during pruning and tuning.

This task locks the model family, target formulation, hyperparameters, and feature-selection count before final evaluation.


In [ ]:
TUNING_OPTIONS = {
    "Ridge": [
        {"alpha": 1.0},
        {"alpha": 10.0},
        {"alpha": 50.0},
    ],
    "Random Forest": [
        {
            "n_estimators": 300,
            "min_samples_leaf": 3,
            "max_features": "sqrt",
        },
        {
            "n_estimators": 400,
            "min_samples_leaf": 5,
            "max_features": 0.7,
        },
        {
            "n_estimators": 500,
            "min_samples_leaf": 8,
            "max_features": 1.0,
        },
    ],
    "XGBoost": [
        {
            "n_estimators": 300,
            "max_depth": 3,
            "learning_rate": 0.05,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_lambda": 1.0,
        },
        {
            "n_estimators": 450,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_lambda": 1.0,
        },
        {
            "n_estimators": 350,
            "max_depth": 5,
            "learning_rate": 0.03,
            "subsample": 0.85,
            "colsample_bytree": 0.9,
            "reg_lambda": 5.0,
        },
    ],
}

# The last eight pre-holdout quarters are used consistently for tuning.
tuning_periods = validation_periods[-8:]

benchmark_tuning_mae = float(
    rolling_results.loc[
        rolling_results["model"].eq(STRONGEST_BENCHMARK)
        & rolling_results["validation_period"].isin(tuning_periods),
        "MAE",
    ].mean()
)

untuned_tuning_mae = float(
    rolling_results.loc[
        rolling_results["model"].eq(WINNING_ML_NAME)
        & rolling_results["validation_period"].isin(tuning_periods),
        "MAE",
    ].mean()
)

untuned_tuning_skill = (
    1 - untuned_tuning_mae / benchmark_tuning_mae
    if benchmark_tuning_mae > 0
    else np.nan
)

tuning_rows = []

for feature_count in FEATURE_SELECTION_K_OPTIONS:
    for params in TUNING_OPTIONS[WINNING_SPEC["family"]]:
        fold_maes = []

        for validation_period in tuning_periods:
            fold_train = pre_holdout.loc[
                pre_holdout["period_index"] < validation_period
            ].copy()
            fold_test = pre_holdout.loc[
                pre_holdout["period_index"] == validation_period
            ].copy()

            fold_features, _, _ = select_features(
                fold_train,
                WINNING_SPEC["target_mode"],
                k=feature_count,
            )

            estimator = clone(
                WINNING_SPEC["estimator"]
            ).set_params(**params)

            prediction = fit_predict(
                fold_train,
                fold_test,
                WINNING_SPEC,
                fold_features,
                estimator=estimator,
            )["prediction"]

            fold_maes.append(
                mean_absolute_error(
                    fold_test[TARGET],
                    prediction,
                )
            )

        mean_validation_mae = float(np.mean(fold_maes))
        tuning_rows.append({
            "feature_count": int(feature_count),
            "params": params,
            "mean_validation_MAE": mean_validation_mae,
            "mean_persistence_MAE": benchmark_tuning_mae,
            "skill_vs_persistence": (
                1 - mean_validation_mae / benchmark_tuning_mae
                if benchmark_tuning_mae > 0
                else np.nan
            ),
            "MAE_improvement_vs_untuned": (
                untuned_tuning_mae - mean_validation_mae
            ),
        })

tuning_results = (
    pd.DataFrame(tuning_rows)
    .sort_values(
        ["mean_validation_MAE", "feature_count"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

BEST_PARAMS = dict(tuning_results.iloc[0]["params"])
BEST_FEATURE_COUNT = int(tuning_results.iloc[0]["feature_count"])
best_tuned_mae = float(tuning_results.iloc[0]["mean_validation_MAE"])
best_tuned_skill = float(tuning_results.iloc[0]["skill_vs_persistence"])

display(tuning_results.round(4))
print(
    "Strongest persistence mean MAE on tuning quarters:",
    round(benchmark_tuning_mae, 2),
)
print(
    "Untuned mean MAE on the same tuning quarters:",
    round(untuned_tuning_mae, 2),
)
print(
    "Untuned skill vs persistence:",
    round(untuned_tuning_skill, 3),
)
print(
    "Best tuned mean MAE on those quarters:",
    round(best_tuned_mae, 2),
)
print(
    "Best tuned skill vs persistence:",
    round(best_tuned_skill, 3),
)
print(
    "MAE reduction from untuned to tuned:",
    round(untuned_tuning_mae - best_tuned_mae, 2),
)
print("Best feature count:", BEST_FEATURE_COUNT)
print("Best parameters:", BEST_PARAMS)

tuning_results.assign(
    params=tuning_results["params"].astype(str)
).to_csv(
    PROCESSED_DIR / "tuning_results.csv",
    index=False,
)


### Task 8 — Locked final holdout evaluation

The locked final holdout is evaluated only after the ML family, target formulation, feature-selection procedure, feature count, and tuning choices are fixed.

The evaluation compares the selected ML model with both quarterly persistence benchmarks and reports out-of-time MAE, RMSE, WAPE, level R², change-based R² relative to both previous-quarter and seasonal persistence, and skill versus the strongest persistence benchmark. Error is also reviewed by quarter, region, and county.


In [ ]:
final_features, _, final_feature_report = select_features(
    holdout_train,
    WINNING_SPEC["target_mode"],
    k=BEST_FEATURE_COUNT,
)

final_estimator = clone(
    WINNING_SPEC["estimator"]
).set_params(**BEST_PARAMS)

final_fit = fit_predict(
    holdout_train,
    holdout_test,
    WINNING_SPEC,
    final_features,
    estimator=final_estimator,
)

ml_prediction = final_fit["prediction"]
actual = holdout_test[TARGET].to_numpy()
lag1_prediction = holdout_test["target_lag1"].to_numpy()
lag4_prediction = holdout_test["target_lag4"].to_numpy()

prediction_map = {
    "Previous-quarter persistence": lag1_prediction,
    "Seasonal persistence": lag4_prediction,
    f"Tuned {WINNING_ML_NAME}": ml_prediction,
}

benchmark_prediction = (
    lag1_prediction
    if STRONGEST_BENCHMARK == "Previous-quarter persistence"
    else lag4_prediction
)

comparison_rows = []
for model_name, prediction in prediction_map.items():
    comparison_rows.append({
        "model": model_name,
        **regression_metrics(actual, prediction),
        "R2_change_from_previous_quarter": change_r2(
            actual,
            prediction,
            lag1_prediction,
        ),
        "R2_change_from_seasonal_persistence": change_r2(
            actual,
            prediction,
            lag4_prediction,
        ),
        "skill_vs_strongest_persistence": (
            0.0
            if model_name == STRONGEST_BENCHMARK
            else skill_vs(
                actual,
                prediction,
                benchmark_prediction,
            )
        ),
    })

final_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(final_comparison.round(4))
final_comparison.to_csv(
    PROCESSED_DIR / "final_holdout_comparison.csv",
    index=False,
)
final_feature_report.to_csv(
    PROCESSED_DIR / "final_feature_selection.csv",
    index=False,
)

holdout_predictions = holdout_test[
    [
        "fips",
        "county_name",
        "year",
        "quarter",
        "quarter_num",
        "region",
        TARGET,
    ]
].copy()

holdout_predictions["previous_quarter_persistence"] = lag1_prediction
holdout_predictions["seasonal_persistence"] = lag4_prediction
holdout_predictions["ml_prediction"] = ml_prediction
holdout_predictions["ml_residual"] = (
    holdout_predictions[TARGET]
    - holdout_predictions["ml_prediction"]
)
holdout_predictions["ml_absolute_error"] = (
    holdout_predictions["ml_residual"].abs()
)
holdout_predictions["benchmark_prediction"] = benchmark_prediction
holdout_predictions["benchmark_residual"] = (
    holdout_predictions[TARGET]
    - holdout_predictions["benchmark_prediction"]
)
holdout_predictions["benchmark_absolute_error"] = (
    holdout_predictions["benchmark_residual"].abs()
)
holdout_predictions.to_csv(
    PROCESSED_DIR / "holdout_predictions.csv",
    index=False,
)

holdout_method_predictions = pd.concat(
    [
        holdout_predictions.assign(
            method=f"Tuned {WINNING_ML_NAME}",
            prediction=holdout_predictions["ml_prediction"],
            residual=holdout_predictions["ml_residual"],
            absolute_error=holdout_predictions["ml_absolute_error"],
        ),
        holdout_predictions.assign(
            method=STRONGEST_BENCHMARK,
            prediction=holdout_predictions["benchmark_prediction"],
            residual=holdout_predictions["benchmark_residual"],
            absolute_error=holdout_predictions["benchmark_absolute_error"],
        ),
    ],
    ignore_index=True,
)

error_by_quarter = (
    holdout_method_predictions.groupby(
        ["method", "quarter"],
        as_index=False,
    )
    .agg(
        rows=(TARGET, "size"),
        MAE=("absolute_error", "mean"),
        mean_residual=("residual", "mean"),
    )
)

error_by_region = (
    holdout_method_predictions.groupby(
        ["method", "region"],
        as_index=False,
    )
    .agg(
        rows=(TARGET, "size"),
        MAE=("absolute_error", "mean"),
        mean_residual=("residual", "mean"),
    )
)

error_by_county = (
    holdout_predictions.groupby(
        ["fips", "county_name"],
        as_index=False,
    )
    .agg(
        quarters=(TARGET, "size"),
        MAE=("ml_absolute_error", "mean"),
        mean_residual=("ml_residual", "mean"),
    )
    .sort_values("MAE", ascending=False)
)

print("Holdout comparison by quarter:")
display(error_by_quarter.round(2))

print("Holdout comparison by region:")
display(error_by_region.round(2))

print("Counties with the largest ML holdout MAE:")
display(error_by_county.head(10).round(2))

error_by_quarter.to_csv(
    PROCESSED_DIR / "holdout_error_by_quarter.csv",
    index=False,
)
error_by_region.to_csv(
    PROCESSED_DIR / "holdout_error_by_region.csv",
    index=False,
)
error_by_county.to_csv(
    PROCESSED_DIR / "holdout_error_by_county.csv",
    index=False,
)

holdout_plot_data = holdout_predictions.assign(
    actual=holdout_predictions[TARGET],
    predicted=holdout_predictions["ml_prediction"],
    absolute_error=holdout_predictions["ml_absolute_error"],
)
holdout_plot_data["observation_label"] = (
    holdout_plot_data["county_name"].astype(str)
    + " — "
    + holdout_plot_data["year"].astype(int).astype(str)
    + " "
    + holdout_plot_data["quarter"].astype(str)
)

fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(
    holdout_plot_data["actual"],
    holdout_plot_data["predicted"],
    alpha=0.55,
)
lower = min(
    holdout_plot_data["actual"].min(),
    holdout_plot_data["predicted"].min(),
)
upper = max(
    holdout_plot_data["actual"].max(),
    holdout_plot_data["predicted"].max(),
)
ax.plot([lower, upper], [lower, upper], linestyle="--")
add_selected_scatter_labels(
    ax,
    holdout_plot_data,
    x_column="actual",
    y_column="predicted",
    label_column="observation_label",
    score_column="absolute_error",
    n_labels=10,
    x_formatter=format_count,
    y_formatter=format_count,
    note="Labels show the 10 largest prediction errors.",
)
ax.set_title(f"Actual vs Predicted ED Encounters, {HOLDOUT_YEAR}")
ax.set_xlabel("Actual ED encounters")
ax.set_ylabel("Predicted ED encounters")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(
    IMAGE_DIR / "actual_vs_predicted.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    holdout_plot_data["predicted"],
    holdout_predictions["ml_residual"],
    alpha=0.55,
)
ax.axhline(0, linestyle="--", linewidth=1)

residual_plot_data = holdout_plot_data.assign(
    residual=holdout_predictions["ml_residual"],
)
add_selected_scatter_labels(
    ax,
    residual_plot_data,
    x_column="predicted",
    y_column="residual",
    label_column="observation_label",
    score_column="absolute_error",
    n_labels=10,
    x_formatter=format_count,
    y_formatter=lambda value: f"{value:+,.0f}",
    note="Labels show the 10 largest absolute residuals.",
)
ax.set_title(f"Holdout Residuals, {HOLDOUT_YEAR}")
ax.set_xlabel("Predicted ED encounters")
ax.set_ylabel("Residual: actual minus prediction")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(
    IMAGE_DIR / "holdout_residuals.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

print("Final selected features:", final_features)


### Holdout diagnostic interpretation

In the actual-versus-predicted chart, the dashed 45-degree line represents perfect predictions. Points below the line are underpredictions and points above it are overpredictions. The labeled points are separate county-quarter observations, so the same county can appear several times.

Residuals are defined as `actual - prediction`. Positive residuals therefore mean underprediction, while negative residuals mean overprediction. Residuals are defined as actual - prediction. Positive residuals indicate underprediction, while negative residuals indicate overprediction. The largest absolute errors are concentrated in high-volume counties, particularly New York City and several nearby counties. New York City is underpredicted on average, while downstate non-NYC counties include both substantial underpredictions and overpredictions. This pattern is consistent with regression toward the mean in the selected tree-based XGBoost model, which generally does not extrapolate large new demand levels beyond patterns represented in training.

### Task 9 — Model interpretation and error diagnosis

SHAP describes the selected machine-learning model, not the persistence benchmark.

- For a level model, SHAP explains the predicted encounter level.
- For a change model, SHAP explains the predicted deviation from `target_lag4`; the seasonal persistence value is added afterward to reconstruct the level.
- Global importance and beeswarm plots summarize overall feature influence.
- A local waterfall explains the largest holdout error so interpretation is connected to model risk rather than only average behavior.

Tree models use `tree_path_dependent` SHAP without a background dataset, which avoids the categorical-split compatibility error seen with XGBoost.

The SHAP importance bar chart reports `mean(|SHAP value|)`: average impact magnitude in model-output units, not a positive causal effect. In the beeswarm, horizontal position gives the direction and size of the contribution, while color represents the observed feature value. Because demand lags and population are strongly correlated, attribution can be shared among them and their exact ranking should not be interpreted as independent causal importance.

In the local waterfall, red features push the prediction above the model's average output and blue features push it below. The waterfall explains why the model produced its prediction; it does not explain the actual outcome or prove causality.


In [ ]:
X_train_shap = holdout_train[final_features].astype(float)
X_test_shap = holdout_test[final_features].astype(float)

if final_fit["scaler"] is not None:
    X_train_shap = pd.DataFrame(
        final_fit["scaler"].transform(X_train_shap),
        columns=final_features,
        index=X_train_shap.index,
    )
    X_test_shap = pd.DataFrame(
        final_fit["scaler"].transform(X_test_shap),
        columns=final_features,
        index=X_test_shap.index,
    )

X_explain = X_test_shap.sample(
    min(150, len(X_test_shap)),
    random_state=42,
)

model_for_shap = final_fit["model"]

if WINNING_SPEC["family"] == "Ridge":
    background = X_train_shap.sample(
        min(200, len(X_train_shap)),
        random_state=42,
    )
    explainer = shap.LinearExplainer(
        model_for_shap,
        background,
    )
    shap_values = explainer(X_explain)
else:
    explainer = shap.TreeExplainer(
        model_for_shap,
        feature_perturbation="tree_path_dependent",
        model_output="raw",
    )
    shap_values = explainer(
        X_explain,
        check_additivity=False,
    )

shap_importance = pd.DataFrame({
    "feature": final_features,
    "mean_abs_shap": np.abs(shap_values.values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance.head(15).round(3))
shap_importance.to_csv(
    PROCESSED_DIR / "shap_feature_importance.csv",
    index=False,
)

shap.plots.bar(
    shap_values,
    max_display=15,
    show=False,
)
shap_bar_ax = plt.gca()
if not shap_bar_ax.texts:
    for patch in shap_bar_ax.patches:
        width = patch.get_width()
        if np.isfinite(width) and width > 0:
            shap_bar_ax.annotate(
                format_decimal(width, 2),
                xy=(width, patch.get_y() + patch.get_height() / 2),
                xytext=(4, 0),
                textcoords="offset points",
                ha="left",
                va="center",
                fontsize=8,
            )
plt.title(f"SHAP Importance: {WINNING_ML_NAME}")
plt.tight_layout()
plt.savefig(
    IMAGE_DIR / "shap_importance.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

shap.plots.beeswarm(
    shap_values,
    max_display=15,
    show=False,
)
plt.title(f"SHAP Feature Effects: {WINNING_ML_NAME}")
plt.tight_layout()
plt.savefig(
    IMAGE_DIR / "shap_beeswarm.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

largest_error_index = holdout_predictions["ml_absolute_error"].idxmax()
X_local = X_test_shap.loc[[largest_error_index]]

if WINNING_SPEC["family"] == "Ridge":
    local_shap_values = explainer(X_local)
else:
    local_shap_values = explainer(
        X_local,
        check_additivity=False,
    )

print("Largest holdout error:")
display(
    holdout_predictions.loc[
        [largest_error_index],
        [
            "county_name",
            "year",
            "quarter",
            TARGET,
            "ml_prediction",
            "ml_residual",
        ],
    ]
)

shap.plots.waterfall(
    local_shap_values[0],
    max_display=15,
    show=False,
)
plt.title("SHAP Explanation for the Largest Holdout Error")
plt.tight_layout()
plt.savefig(
    IMAGE_DIR / "shap_largest_error_waterfall.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


### Task 10 — Inference validation, saved artifact, model card, and final reporting

The saved artifact contains the target definition, selected features, scaler when required, fitted ML model, target formulation, validation choice, final metrics, training and holdout periods, data snapshot date, and recommended operational method.

Three small prediction functions are validated separately:

- the tuned machine-learning path;
- previous-quarter and seasonal persistence;
- the final recommended-method path.

This ensures that the serialized ML model is tested even when persistence remains the operational recommendation.


In [ ]:
ml_model_label = f"Tuned {WINNING_ML_NAME}"

ml_row = final_comparison.loc[
    final_comparison["model"].eq(ml_model_label)
].iloc[0]

benchmark_row = final_comparison.loc[
    final_comparison["model"].eq(STRONGEST_BENCHMARK)
].iloc[0]

if ml_row["MAE"] < benchmark_row["MAE"]:
    RECOMMENDED_METHOD = ml_model_label
    recommendation = (
        f"The tuned ML model reduced holdout MAE from "
        f"{benchmark_row['MAE']:,.0f} to {ml_row['MAE']:,.0f}. "
        "Use it as the preferred prototype forecast and monitor "
        "performance after each new quarter."
    )
else:
    RECOMMENDED_METHOD = STRONGEST_BENCHMARK
    recommendation = (
        f"The tuned ML model did not beat {STRONGEST_BENCHMARK} "
        f"on the final holdout. Benchmark MAE was "
        f"{benchmark_row['MAE']:,.0f}; ML MAE was "
        f"{ml_row['MAE']:,.0f}. Use persistence as the primary "
        "prototype forecast and retain ML for comparison and diagnostics."
    )

print("Recommended method:", RECOMMENDED_METHOD)
print(recommendation)


def _validated_numeric_columns(frame, columns, label):
    missing_columns = [column for column in columns if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"Missing required {label} columns: {missing_columns}")

    try:
        numeric = frame.loc[:, columns].apply(pd.to_numeric, errors="raise")
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{label.capitalize()} columns must be numeric-compatible."
        ) from exc

    columns_with_missing = numeric.columns[numeric.isna().any()].tolist()
    if columns_with_missing:
        raise ValueError(
            f"Missing values found in {label} columns: {columns_with_missing}"
        )

    if not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise ValueError(f"Non-finite values found in {label} columns.")

    return numeric


def predict_ml_quarterly_ed(input_frame, model_artifact):
    """Return the saved machine-learning model's level prediction."""
    if not isinstance(input_frame, pd.DataFrame):
        raise TypeError("input_frame must be a pandas DataFrame.")

    features = list(model_artifact["features"])
    X = _validated_numeric_columns(input_frame, features, "model feature")

    scaler = model_artifact["scaler"]
    X_fit = scaler.transform(X) if scaler is not None else X
    component_prediction = model_artifact["model"].predict(X_fit)

    if model_artifact["target_mode"] == "change":
        lag = _validated_numeric_columns(
            input_frame,
            ["target_lag4"],
            "reconstruction lag",
        )
        return lag["target_lag4"].to_numpy() + component_prediction

    return np.asarray(component_prediction)


def predict_persistence(input_frame, method):
    """Return a previous-quarter or seasonal persistence prediction."""
    lag_column = {
        "Previous-quarter persistence": "target_lag1",
        "Seasonal persistence": "target_lag4",
    }.get(method)

    if lag_column is None:
        raise ValueError(f"Unknown persistence method: {method}")

    lag = _validated_numeric_columns(
        input_frame,
        [lag_column],
        "persistence",
    )
    return lag[lag_column].to_numpy()


def predict_recommended_quarterly_ed(input_frame, model_artifact):
    """Return the prediction from the artifact's recommended method."""
    method = model_artifact["recommended_method"]

    if method in {
        "Previous-quarter persistence",
        "Seasonal persistence",
    }:
        return predict_persistence(input_frame, method)

    return predict_ml_quarterly_ed(input_frame, model_artifact)


train_periods = holdout_train.sort_values("period_index")
train_start_period = (
    f"{int(train_periods.iloc[0]['year'])} "
    f"{train_periods.iloc[0]['quarter']}"
)
train_end_period = (
    f"{int(train_periods.iloc[-1]['year'])} "
    f"{train_periods.iloc[-1]['quarter']}"
)

artifact = {
    "project": "New York quarterly facility-county ED forecasting",
    "target": TARGET,
    "target_definition": (
        "Total ED encounters aggregated by facility county and quarter"
    ),
    "unit_of_analysis": "facility county-quarter",
    "forecast_horizon": "sequential one quarter ahead",
    "model_name": WINNING_ML_NAME,
    "target_mode": WINNING_SPEC["target_mode"],
    "feature_selection_k": BEST_FEATURE_COUNT,
    "features": final_features,
    "scaler": final_fit["scaler"],
    "model": final_fit["model"],
    "best_params": BEST_PARAMS,
    "train_start_period": train_start_period,
    "train_end_period": train_end_period,
    "training_rows": int(len(holdout_train)),
    "holdout_year": int(HOLDOUT_YEAR),
    "holdout_rows": int(len(holdout_test)),
    "data_as_of_date": DATA_AS_OF_DATE,
    "strongest_benchmark": STRONGEST_BENCHMARK,
    "recommended_method": RECOMMENDED_METHOD,
    "final_metrics": final_comparison.to_dict(orient="records"),
    "versions": {
        "python": sys.version.split()[0],
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "shap": shap.__version__,
    },
}

artifact_path = MODEL_DIR / "quarterly_ed_forecast_artifact.joblib"
joblib.dump(artifact, artifact_path)

# Verify each prediction path independently.
verification_rows = holdout_test.head(3)
ml_check = predict_ml_quarterly_ed(verification_rows, artifact)
lag1_check = predict_persistence(
    verification_rows,
    "Previous-quarter persistence",
)
lag4_check = predict_persistence(
    verification_rows,
    "Seasonal persistence",
)
recommended_check = predict_recommended_quarterly_ed(
    verification_rows,
    artifact,
)

print("ML inference check:", np.round(ml_check, 2))
print("Previous-quarter persistence check:", np.round(lag1_check, 2))
print("Seasonal persistence check:", np.round(lag4_check, 2))
print("Recommended-method check:", np.round(recommended_check, 2))

model_card_text = f"""
# Model Card: New York Quarterly ED Demand

## Target
Total ED encounters aggregated by facility county and quarter.

## Forecasting use
Retrospective one-quarter-ahead forecasting prototype for staffing and capacity analysis.

## Data period
Facility target years: {FACILITY_START_YEAR}–{FACILITY_END_YEAR}.
SPARCS snapshot date: {DATA_AS_OF_DATE}.
Final holdout year: {HOLDOUT_YEAR}.

## Validation
Expanding-window rolling validation by complete quarter.
Final evaluation on the latest complete holdout year.

## Benchmarks
Previous-quarter persistence and same-quarter previous-year persistence.
Strongest pre-holdout benchmark: {STRONGEST_BENCHMARK}.

## Selected machine-learning model
{WINNING_ML_NAME}

## Selected feature count
{BEST_FEATURE_COUNT}

## Recommended operational method
{RECOMMENDED_METHOD}

## Important limitations
- Facility county is not necessarily patient residence.
- The per-capita measure is descriptive and is not the forecasting target.
- The final holdout covers one calendar year.
- Publicly suppressed facility cells may understate affected facility county-quarter totals by an unknown amount.
- Pooled R² is secondary because county volumes differ greatly in scale.
"""

model_card_path = REPORT_DIR / "model_card.md"
model_card_path.write_text(
    model_card_text.strip() + "\n",
    encoding="utf-8",
)

# Build a self-contained final model-versus-persistence table
# for the one-page stakeholder results summary.
summary_metrics = final_comparison[
    [
        "model",
        "MAE",
        "RMSE",
        "WAPE",
        "R2",
        "skill_vs_strongest_persistence",
    ]
].copy()

summary_metrics["WAPE_percent"] = summary_metrics["WAPE"] * 100
summary_metrics = summary_metrics[
    [
        "model",
        "MAE",
        "RMSE",
        "WAPE_percent",
        "R2",
        "skill_vs_strongest_persistence",
    ]
].rename(
    columns={
        "model": "Method",
        "WAPE_percent": "WAPE (%)",
        "R2": "Level R²",
        "skill_vs_strongest_persistence": "Skill",
    }
)


def format_summary_value(value, decimals=2):
    if pd.isna(value):
        return "NA"
    return f"{float(value):,.{decimals}f}"


summary_table_lines = [
    "| Method | MAE | RMSE | WAPE (%) | Level R² | Skill |",
    "|---|---:|---:|---:|---:|---:|",
]

for _, row in summary_metrics.iterrows():
    summary_table_lines.append(
        "| "
        + str(row["Method"])
        + " | "
        + format_summary_value(row["MAE"], 1)
        + " | "
        + format_summary_value(row["RMSE"], 1)
        + " | "
        + format_summary_value(row["WAPE (%)"], 2)
        + " | "
        + format_summary_value(row["Level R²"], 4)
        + " | "
        + format_summary_value(row["Skill"], 4)
        + " |"
    )

summary_metrics_markdown = "\n".join(summary_table_lines)

summary_text = f"""
# Quarterly ED Demand Results

- SPARCS snapshot date: {DATA_AS_OF_DATE}
- Final holdout year: {HOLDOUT_YEAR}
- Selected ML candidate: {WINNING_ML_NAME}
- Target formulation: {WINNING_SPEC["target_mode"]}
- Selected feature count: {BEST_FEATURE_COUNT}
- Strongest persistence benchmark: {STRONGEST_BENCHMARK}
- Recommended operational method: {RECOMMENDED_METHOD}
- Selected features: {", ".join(final_features)}

## Final holdout comparison

{summary_metrics_markdown}

Skill is measured against {STRONGEST_BENCHMARK}. Positive skill
means the method reduces MAE relative to that benchmark; negative
skill means it performs worse.

## Week of July 14 — model family carried forward

{carry_forward_paragraph}

## Week of July 21 — when the model adds value over persistence

{rolling_value_paragraph}

## Final recommendation

{recommendation}

## Important limitations

- Facility county is not necessarily patient county of residence.
- The final holdout contains only one complete calendar year.
- Publicly suppressed facility cells may understate affected totals by an unknown amount.
- The largest absolute errors occur in high-volume urban counties.
- Pooled level R² is secondary to out-of-time MAE and skill.
- Persistence remains the operational recommendation whenever it
  outperforms the machine-learning model.
"""

results_summary_path = REPORT_DIR / "results_summary.md"
results_summary_path.write_text(
    summary_text.strip() + "\n",
    encoding="utf-8",
)


# Create a compact output manifest for the final submission.
manifest_path = REPORT_DIR / "output_manifest.csv"
manifest_rows = []

for category, folder in {
    "processed_data": PROCESSED_DIR,
    "image": IMAGE_DIR,
    "model": MODEL_DIR,
    "report": REPORT_DIR,
}.items():
    for file_path in sorted(folder.glob("*")):
        if file_path.is_file() and file_path != manifest_path:
            manifest_rows.append({
                "category": category,
                "file": file_path.name,
                "relative_path": str(file_path.relative_to(PROJECT_DIR)),
                "size_bytes": int(file_path.stat().st_size),
            })

output_manifest = pd.DataFrame(manifest_rows).sort_values(
    ["category", "file"]
)
output_manifest.to_csv(manifest_path, index=False)

print("Saved model artifact:", artifact_path)
print("Saved model card:", model_card_path)
print("Saved results summary:", results_summary_path)
print("Saved output manifest:", manifest_path)


### Submission checklist — scope through July 28

- [x] **Phase 1: Configuration**
- [x] **Phase 2 — Task 1: Data Collection & Acquisition**
- [x] **Phase 3 — Task 2: Data Cleaning & Preprocessing**
- [x] **Phase 4 — Task 3: Exploratory Data Analysis & Visualization**
- [x] **Phase 5 — Task 4: Feature Engineering & Selection**
- [x] **Week of July 14 — Task 5: Model Selection and Development**
- [x] **Week of July 21 — Task 6: Rolling-Origin Evaluation and Validation, including a visible skill-by-quarter table**
- [x] **Phase 6 / Week of July 28 — Task 7: Feature Pruning and Time-Aware Tuning**
- [x] **Phase 6 / Week of July 28 — Task 8: Locked Final Holdout Evaluation**
- [x] **Phase 6 / Week of July 28 — Task 9: SHAP Interpretation and Error Diagnosis**
- [x] **Phase 6 / Week of July 28 — Task 10: Artifact, Inference Validation, Model Card, and final model-versus-persistence results summary**

### Week of July 28 summary

1. Tunes only the ML family supported by rolling-origin validation.
2. Keeps feature pruning and feature-count selection inside time-aware folds.
3. Preserves the latest complete year as a locked final holdout.
4. Compares the final ML model with previous-quarter and seasonal persistence.
5. Reports whether ML genuinely improves out-of-time MAE; persistence remains the operational recommendation when it is stronger.
6. Produces global and local SHAP explanations for the selected ML model.
7. Saves a documented artifact only after the target, features, model, and evaluation procedure are locked.
8. Validates the ML, persistence, and recommended prediction paths and creates the model card, output manifest, visible quarterly skill table, and stakeholder-facing results summary with final MAE, RMSE, WAPE, R², and skill.

## Final interpretation

The central stakeholder question is not whether pooled R² is close to 1. The decision is whether the selected method reduces out-of-time MAE relative to simple persistence and whether that improvement is stable across quarters and regions.

A defensible final presentation should report:

- the latest pre-holdout statewide demand pattern and quarterly seasonality;
- the operational target and the per-capita EDA caveat;
- the strongest persistence benchmark;
- rolling MAE, skill by quarter, and the number of quarters ML wins;
- final holdout MAE, RMSE, WAPE, level R², and change-based R²;
- error by quarter, region, and county;
- selected features and their stability;
- SHAP global and local explanations;
- the final operational recommendation and limitations.


The public-data app should be presented as a retrospective prototype. A live operational version would require a current internal encounter feed.

#### Check versionss for the app

In [ ]:
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost

print("Python:", sys.version)
print("XGBoost:", xgboost.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

In [ ]:
from pathlib import Path
import joblib

MODEL_DIR = Path(
    "/content/drive/MyDrive/MS DATA SCIENCE/Capstone975/model"
)

# Save XGBoost in its portable format.
artifact["model"].save_model(
    MODEL_DIR / "quarterly_ed_xgboost_model.json"
)

# Save the other information separately.
app_artifact = artifact.copy()
app_artifact.pop("model")

joblib.dump(
    app_artifact,
    MODEL_DIR / "quarterly_ed_forecast_artifact.joblib"
)

print("Files saved successfully.")

In [ ]:
print(type(artifact["model"]))